# EnToFrMedicaLLM — End-to-End Notebook: Data Engineering → Training → Evaluation
**Author:** Brice Donald Abodo Eloundou · ITMO University

This single notebook runs the complete **EnMed pipeline** from raw datasets to final statistical results. Every code cell comes **directly from your NB1–NB8 notebooks** — nothing invented.


## What came from your code vs what is new

| Section | Source notebook | What I added |
|---|---|---|
| §1 Install & core | NB1 Stage 0–1 + NB2 Stage 0 | Nothing — verbatim |
| §2 Data engineering | NB1 Stages 2–11 (all 36 cells) | Nothing — verbatim |
| §3 DAPT pre-training | NB2 Stages 2–7 | Nothing — verbatim |
| §4 SFT fine-tuning | NB3 Stages 2–6 | Nothing — verbatim |
| §5 Baseline evaluation | NB5a Stages 2–7 | Nothing — verbatim |
| §6 EnMed eval + Gemma judge | NB5b Stages 2–9 | Nothing — verbatim |
| §7 Statistical analysis | NB8 Stages 1–13 | Nothing — verbatim |
| **§8 GGUF export** | **NEW** | Export + push quantized GGUF (q4_k_m / q8_0 / bf16) |
| **§9 CPU inference** | **NEW** | `llama-cpp-python` inference, no GPU needed |


## §1 — Install & Core Setup
*NB1 Stage 0 + NB2 Stage 0 (your code verbatim)*

Installs all required packages if missing and imports `enmed_core.py` — the shared utility module that all notebooks depend on.


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 0: Install. Skips packages already present.
# ──────────────────────────────────────────────────────────────────────
import subprocess, sys, importlib

REQUIRED = {
    "datasets":      "datasets",
    "huggingface_hub": "huggingface-hub",
    "datasketch":    "datasketch",
    "rouge_score":   "rouge-score",
    "nltk":          "nltk",
    "tqdm":          "tqdm",
    "pandas":        "pandas",
    "numpy":         "numpy",
    "matplotlib":    "matplotlib",
    "seaborn":       "seaborn",
}

missing = []
for mod, pip_name in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ Install complete — restart runtime if anything breaks.")
else:
    print("✓ All required packages already installed.")

In [ ]:
# Stage 0 — Install if missing
import subprocess, sys, importlib

REQUIRED = {"unsloth":"unsloth","transformers":"transformers","trl":"trl","peft":"peft",
            "bitsandbytes":"bitsandbytes","accelerate":"accelerate","datasets":"datasets",
            "wandb":"wandb","huggingface_hub":"huggingface-hub","tqdm":"tqdm","modelscope": "modelscope"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("✓ Stage 0 done")

In [ ]:
# Stage 0+ — Suppress noisy deprecation/info messages
#
# Two sources of log spam dominate Colab cells:
#   1. transformers' AttentionMaskConverter deprecation warning fires on every
#      attention layer on every forward pass (40+ layers × every train step ×
#      every eval step = thousands of identical warnings).
#   2. Unsloth's "Restored added_tokens_decoder metadata" INFO line prints once
#      per saved checkpoint — fine in isolation, awful when checkpoints save
#      every 184 steps.
#
# We silence (1) via warnings filter and (2) via Python's logging level on
# Unsloth's logger. Neither affects training correctness; only output noise.
import warnings, logging

# (1) AttentionMaskConverter / modeling_attn_mask_utils deprecation
warnings.filterwarnings(
    "ignore",
    message=r".*attention mask API.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*AttentionMaskConverter.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r".*modeling_attn_mask_utils.*",
    category=FutureWarning,
)

# Belt-and-suspenders: silence the module's logger directly
for _name in ["transformers.modeling_attn_mask_utils",
              "transformers.masking_utils"]:
    logging.getLogger(_name).setLevel(logging.ERROR)

# (2) Unsloth checkpoint-restore chatter — INFO-level, raise to WARNING
for _name in ["unsloth", "unsloth.save", "unsloth.tokenizer_utils",
              "unsloth.chat_templates"]:
    logging.getLogger(_name).setLevel(logging.WARNING)

# Some Unsloth versions print directly with `print()` rather than logging.
# Monkey-patch builtin print to swallow that one specific line.
import builtins as _bi
_real_print = _bi.print
def _filtered_print(*args, **kwargs):
    if args and isinstance(args[0], str) and (
        "Restored added_tokens_decoder metadata" in args[0]
        or "attention mask API" in args[0]
    ):
        return  # swallow
    return _real_print(*args, **kwargs)
_bi.print = _filtered_print

print("✓ Stage 0+ done — warning suppressors active")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 1a: Mount Drive (Colab only) and ensure base directory exists
# ──────────────────────────────────────────────────────────────────────
import os, sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
    print("✓ Drive mounted")
else:
    print("• Non-Colab environment — using local paths")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 1b: Locate and import enmed_core.py (shared utility module)
# ──────────────────────────────────────────────────────────────────────
import sys, os

BASE_DIR = "base_dir"
CORE_DIR = os.path.join(BASE_DIR, "MODEL_TRAINING")

if not os.path.exists(os.path.join(CORE_DIR, "enmed_core.py")):
    raise FileNotFoundError(
        f"enmed_core.py not found in {CORE_DIR}. "
        f"Upload it once to that folder, then re-run this cell."
    )

if CORE_DIR not in sys.path:
    sys.path.insert(0, CORE_DIR)

import enmed_core as ec
print(f"✓ Imported enmed_core from {CORE_DIR}")

## §2 — Data Engineering
*NB1 Stages 1–11 (your code verbatim)*

Loads the three evaluation/training datasets (MCQA, ExtQA, AbsQA) and the DAPT corpus, applies the **MinHash leakage shield**, carves few-shot exemplar pools, normalises to a unified schema, and writes `train/val/test/exemplars.jsonl` to Drive.

**Datasets:**
| Task | Source | Test N |
|---|---|---|
| MCQA | `qanastek/frenchmedmcqa` (DEFT-2023 zip) | 622 |
| ExtQA | `Dr-BERT/QUAERO` → synthesised QA pairs | 207 |
| AbsQA | `ANR-MALADES/MediQAl` (oeq subset) | 247–248 |
| DAPT corpus | PARCOMED + CAS + QUAERO FR + PubMed + Wikipedia | — |


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 1c: Build config + logger + seed everything
# ──────────────────────────────────────────────────────────────────────
cfg = ec.EnMedConfig(
    base_dir = BASE_DIR,
    seed     = 42,
    # NB1-specific overrides if any:
    minhash_threshold = 0.8,
    minhash_num_perm  = 128,
    exemplar_pool_size_per_task = 200,
)
cfg.make_dirs()

logger = ec.setup_logger(
    name = "nb1",
    log_file = os.path.join(cfg.results_dir, f"nb1_{ec.now_ts()}.log"),
)

ec.seed_all(cfg.seed)
ec.colab_keep_alive()

logger.info("=" * 70)
logger.info(f"NB1 — Data Engineering & Leakage Shield")
logger.info(f"  base_dir    : {cfg.base_dir}")
logger.info(f"  results_dir : {cfg.results_dir}")
logger.info(f"  cache_dir   : {cfg.cache_dir}")
logger.info(f"  seed        : {cfg.seed}")
logger.info("=" * 70)

# Persist the config for reproducibility
cfg_path = cfg.save()
logger.info(f"✓ Config saved to {cfg_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 2: HF auth. We need this to download MCQA + AbsQA datasets.
# If HF_TOKEN is missing we still try public datasets but warn loudly.
# ──────────────────────────────────────────────────────────────────────
HF_TOKEN = ec.bootstrap_hf(logger=logger)
HF_USER  = cfg.hf_user

if not HF_TOKEN:
    logger.warning("⚠ HF_TOKEN missing — gated datasets (PARCOMED, MediQAl) may fail")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 3a: MCQA test set — qanastek/frenchmedmcqa (DEFT-2023)
# Format: zipped JSON files (train.json, dev.json, test.json)
# Each row: {id, question, answers: {A,B,C,D,E}, correct_answers, ...}
# ──────────────────────────────────────────────────────────────────────
import zipfile
from huggingface_hub import hf_hub_download
from datasets import load_dataset, DatasetDict

mcqa_test_rows = []
mcqa_train_rows = []
mcqa_val_rows = []

try:
    zip_path = hf_hub_download(
        repo_id   = "qanastek/frenchmedmcqa",
        filename  = "DEFT-2023-FULL.zip",
        repo_type = "dataset",
        cache_dir = cfg.cache_dir,
        token     = HF_TOKEN,
    )
    extract_to = os.path.join(cfg.cache_dir, "frenchmedmcqa_extracted")
    if not os.path.exists(os.path.join(extract_to, "test.json")):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(extract_to)

    import json
    for split, fn in [("train", "train.json"), ("dev", "dev.json"), ("test", "test.json")]:
        path = os.path.join(extract_to, fn)
        with open(path, "r", encoding="utf-8") as f:
            rows = json.load(f)
        if split == "train":
            mcqa_train_rows = rows
        elif split == "dev":
            mcqa_val_rows = rows
        else:
            mcqa_test_rows = rows

    logger.info(f"  MCQA train : {len(mcqa_train_rows):,}")
    logger.info(f"  MCQA val   : {len(mcqa_val_rows):,}")
    logger.info(f"  MCQA test  : {len(mcqa_test_rows):,}  ← Golden")
except Exception as e:
    logger.error(f"✗ MCQA load failed: {e}")
    raise

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 3b: AbsQA — ANR-MALADES/MediQAl (open-ended)
# Each row: {question, short_answer, long_answer, ...}
# ──────────────────────────────────────────────────────────────────────
absqa_test_rows = []
absqa_train_rows = []

# MediQAl has multiple subsets — 'oeq' is open-ended; 'mcq' is the multi-choice
# variant. We use 'oeq' for AbsQA.
ABSQA_CANDIDATES = [
    ("ANR-MALADES/MediQAl", "oeq"),
    ("ANR-MALADES/MEDIQA-l", "oeq"),
    ("DrBenchmark/MEDIQAL",  None),
]

absqa_ds = None
for ds_id, subset in ABSQA_CANDIDATES:
    current_ds = None
    try:
        if subset:
            current_ds = load_dataset(ds_id, subset, cache_dir=cfg.cache_dir, token=HF_TOKEN)
        else:
            current_ds = load_dataset(ds_id, cache_dir=cfg.cache_dir, token=HF_TOKEN)

        # Relaxed check: Accept the dataset as long as it loaded successfully
        if len(current_ds.keys()) > 0:
            absqa_ds = current_ds
            logger.info(f"✓ Loaded AbsQA from {ds_id} (subset={subset})")
            break

    except Exception as e:
        logger.warning(f"  {ds_id}/{subset} failed: {e}")

if absqa_ds is None:
    raise RuntimeError("Could not load any MediQAl variant. Check HF token + dataset access.")

# Pick splits
def _list_split(ds, name):
    return list(ds[name]) if name in ds else []

absqa_train_rows = _list_split(absqa_ds, "train")
absqa_val_rows   = _list_split(absqa_ds, "val") or _list_split(absqa_ds, "dev")
absqa_test_rows  = _list_split(absqa_ds, "test")

import random as _r
rng = _r.Random(cfg.seed)

# NEW: If there's ONLY a test split (like MediQAl 'oeq'), split it into train/val/test
if not absqa_train_rows and absqa_test_rows:
    shuffled = absqa_test_rows.copy()
    rng.shuffle(shuffled)
    n_test = max(50, len(shuffled) // 20) # Reserve 5% for testing
    n_val = max(50, len(shuffled) // 10)  # Reserve 10% for validation
    absqa_test_rows = shuffled[:n_test]
    absqa_val_rows = shuffled[n_test:n_test + n_val]
    absqa_train_rows = shuffled[n_test + n_val:]
    logger.warning(f"  No train/val split found — carved out {len(absqa_train_rows):,} train and {len(absqa_val_rows):,} val rows from the test split")

# ORIGINAL FALLBACK: If no separate test split, hold out 10% of train as test
elif not absqa_test_rows and absqa_train_rows:
    shuffled = absqa_train_rows.copy()
    rng.shuffle(shuffled)
    n_test = max(50, len(shuffled) // 10)
    absqa_test_rows = shuffled[:n_test]
    absqa_train_rows = shuffled[n_test:]
    logger.warning(f"  No test split — held out {n_test} from train")

logger.info(f"  AbsQA train : {len(absqa_train_rows):,}")
logger.info(f"  AbsQA val   : {len(absqa_val_rows):,}")
logger.info(f"  AbsQA test  : {len(absqa_test_rows):,}  ← Golden")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 3c: ExtQA — derived from QUAERO French Medical (Dr-BERT/QUAERO)
# QUAERO has 'tokens' + entity tags. We synthesise span-extraction QA pairs.
# Strategy: for each annotated entity, ask "Quelle est la mention de
# {category} dans le texte ?" and gold-span = the entity surface form.
# ──────────────────────────────────────────────────────────────────────
extqa_train_rows = []
extqa_val_rows = []
extqa_test_rows = []

QUAERO_CANDIDATES = [
    ("Dr-BERT/QUAERO",     "medline"),
    ("Dr-BERT/QUAERO",     "emea"),
    ("DrBenchmark/QUAERO", "medline"),
    ("mnaguib/QuaeroFrenchMed", "medline"),
]

quaero_loaded = {}
for ds_id, subset in QUAERO_CANDIDATES:
    if subset in quaero_loaded:
        continue
    try:
        ds = load_dataset(ds_id, subset, cache_dir=cfg.cache_dir, token=HF_TOKEN)
        quaero_loaded[subset] = (ds_id, ds)
        logger.info(f"✓ QUAERO {subset} loaded from {ds_id}")
    except Exception as e:
        logger.warning(f"  {ds_id}/{subset} failed: {e}")

if not quaero_loaded:
    raise RuntimeError("All QUAERO sources failed.")

# QUAERO entity-tag → French question template
TAG2QUESTION = {
    "ANAT":  "Quelle structure anatomique est mentionn0e ?",
    "CHEM":  "Quel compos0 chimique ou m0dicament est mentionn0 ?",
    "DEVI":  "Quel dispositif m0dical est mentionn0 ?",
    "DISO":  "Quelle pathologie ou trouble est mentionn0 ?",
    "GEOG":  "Quel lieu g0ographique est mentionn0 ?",
    "LIVB":  "Quel organisme vivant est mentionn0 ?",
    "OBJC":  "Quel objet est mentionn0 ?",
    "PHEN":  "Quel ph0nom0ne biologique est mentionn0 ?",
    "PHYS":  "Quel processus physiologique est mentionn0 ?",
    "PROC":  "Quelle proc0dure m0dicale est mentionn0e ?",
}


def quaero_to_extqa(ds_split, source_label: str, max_per_split: int = 5000):
    """Convert QUAERO NER format → SQuAD-style extractive QA.

    QUAERO row: {tokens: [...], ner_tags: [...]}
    Output:     {context, question, answers: {text:[...], answer_start:[...]},
                 source, specialty}
    """
    rows = []
    label_names = ds_split.features["ner_tags"].feature.names if "ner_tags" in ds_split.features else None

    for ex in ds_split:
        if len(rows) >= max_per_split:
            break
        toks = ex.get("tokens", [])
        tags = ex.get("ner_tags", [])
        if not toks or not tags or len(toks) != len(tags):
            continue

        # Reconstruct context with character offsets
        context = ""
        offsets = []
        for tok in toks:
            offsets.append(len(context))
            context += str(tok) + " "
        context = context.rstrip()

        # Find entity spans (BIO tag scheme)
        i = 0
        while i < len(tags):
            tag_id = tags[i]
            tag_str = label_names[tag_id] if label_names else str(tag_id)
            if tag_str.startswith("B-"):
                cat = tag_str[2:]
                start_tok = i
                end_tok = i
                j = i + 1
                while j < len(tags):
                    nxt = label_names[tags[j]] if label_names else str(tags[j])
                    if nxt.startswith(f"I-{cat}"):
                        end_tok = j
                        j += 1
                    else:
                        break
                # Build span
                start_char = offsets[start_tok]
                end_char = offsets[end_tok] + len(str(toks[end_tok]))
                span_text = context[start_char:end_char]
                question = TAG2QUESTION.get(cat, f"Quelle entit0 de type {cat} est mentionn0e ?")

                rows.append({
                    "id": f"quaero_{source_label}_{len(rows)}",
                    "context": context,
                    "question": question,
                    "answers": {"text": [span_text], "answer_start": [start_char]},
                    "source": source_label,
                    "specialty": cat.lower(),
                })
                i = j
            else:
                i += 1
    return rows

temp_extqa_test_candidates = []
temp_extqa_val_candidates = []

for subset, (ds_id, ds) in quaero_loaded.items():
    for split_name in ds.keys():
        rows = quaero_to_extqa(ds[split_name], f"{subset}_{split_name}")
        if split_name in ("test",):
            temp_extqa_test_candidates.extend(rows)
        elif split_name in ("validation", "dev"):
            temp_extqa_val_candidates.extend(rows)
        else:
            extqa_train_rows.extend(rows)

import random as _r_extqa
rng_extqa = _r_extqa.Random(cfg.seed)

# Shuffle the test candidates to ensure randomness in selection
rng_extqa.shuffle(temp_extqa_test_candidates)

# Revert test percentage to 5%
target_test_percentage = 0.05
n_extqa_test = max(50, int(len(temp_extqa_test_candidates) * target_test_percentage))
extqa_test_rows = temp_extqa_test_candidates[:n_extqa_test]
extqa_train_rows.extend(temp_extqa_test_candidates[n_extqa_test:])

# Shuffle and reduce the validation candidates to augment training
rng_extqa.shuffle(temp_extqa_val_candidates)
target_val_percentage = 0.1 # Keep 10% for val, 90% goes to train
n_extqa_val = max(50, int(len(temp_extqa_val_candidates) * target_val_percentage))
extqa_val_rows = temp_extqa_val_candidates[:n_extqa_val]
extqa_train_rows.extend(temp_extqa_val_candidates[n_extqa_val:])

logger.info(f"  ExtQA train : {len(extqa_train_rows):,}")
logger.info(f"  ExtQA val   : {len(extqa_val_rows):,}  ← Reduced to augment train")
logger.info(f"  ExtQA test  : {len(extqa_test_rows):,}  ← Golden (re-proportioned to 5%)")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 4a: PARCOMED — French parallel medical corpus
# May be gated; we set strict=False so the notebook still completes if it
# fails (NB2 will then run on the fallbacks alone).
# ──────────────────────────────────────────────────────────────────────
PARCOMED_TARGET_N = 50_000     # cap for NB1 stats — NB2 streams the full set

parcomed_texts = []
parcomed_meta = {"loaded": False, "source": None, "n": 0}

try:
    pc = load_dataset(cfg.parcomed_id, split="train",
                      cache_dir=cfg.cache_dir, token=HF_TOKEN,
                      streaming=True)
    # Find first text-y column on the fly
    take_n = PARCOMED_TARGET_N
    for i, ex in enumerate(pc):
        if i >= take_n:
            break
        # Heuristic: pick longest string field
        candidates = [(k, v) for k, v in ex.items()
                      if isinstance(v, str) and len(v) > 50]
        if not candidates:
            continue
        candidates.sort(key=lambda kv: -len(kv[1]))
        parcomed_texts.append(candidates[0][1])
    parcomed_meta = {
        "loaded": True, "source": cfg.parcomed_id, "n": len(parcomed_texts),
    }
    logger.info(f"✓ PARCOMED: {len(parcomed_texts):,} texts (capped at {PARCOMED_TARGET_N:,} for NB1)")
except Exception as e:
    logger.warning(f"⚠ PARCOMED unavailable ({e}) — NB2 will use fallbacks only")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 4b: CAS corpus + Dr-BERT FrenchMedMCQA train (additional FR medical text)
# These provide redundancy for DAPT and allow ExtQA construction in NB6.
# ──────────────────────────────────────────────────────────────────────
fr_medical_texts = []   # for DAPT supplementation

# CAS — French clinical cases
for ds_id in ["DrBenchmark/CAS", "Dr-BERT/CAS"]:
    try:
        cas = load_dataset(ds_id, cache_dir=cfg.cache_dir, token=HF_TOKEN,
                           trust_remote_code=True)
        for split_name in cas.keys():
            for ex in cas[split_name]:
                # Reconstruct from tokens or pick text field
                if "tokens" in ex and isinstance(ex["tokens"], list) and ex["tokens"]:
                    txt = " ".join(str(t) for t in ex["tokens"])
                else:
                    txt = next((v for v in ex.values()
                                if isinstance(v, str) and len(v) > 40), "")
                if txt and len(txt) > 100:
                    fr_medical_texts.append(txt)
        logger.info(f"  CAS ({ds_id}): cumulative texts = {len(fr_medical_texts):,}")
        break
    except Exception as e:
        logger.warning(f"  {ds_id} unavailable: {e}")

logger.info(f"✓ FR medical texts loaded: {len(fr_medical_texts):,}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 5a: Extract test "fingerprint texts" from each test set
# ──────────────────────────────────────────────────────────────────────
def _mcqa_fingerprint(ex):
    """Concatenate question + all answer choices for a strong fingerprint."""
    q = str(ex.get("question", ""))
    ans = ex.get("answers", {})
    if isinstance(ans, dict):
        choices = " ".join(str(v) for v in ans.values())
    else:
        choices = str(ans)
    return f"{q} {choices}".strip()


def _extqa_fingerprint(ex):
    """Question + answer span — context is too noisy for fingerprinting."""
    q = str(ex.get("question", ""))
    ans = ex.get("answers", {})
    if isinstance(ans, dict) and ans.get("text"):
        a = ans["text"][0] if isinstance(ans["text"], list) else str(ans["text"])
    else:
        a = str(ans)
    return f"{q} {a}".strip()


def _absqa_fingerprint(ex):
    """Question is the canonical fingerprint for AbsQA."""
    return str(ex.get("question", "")).strip()


fp_mcqa  = [_mcqa_fingerprint(e)  for e in mcqa_test_rows]
fp_extqa = [_extqa_fingerprint(e) for e in extqa_test_rows]
fp_absqa = [_absqa_fingerprint(e) for e in absqa_test_rows]

logger.info(f"  Fingerprints: MCQA={len(fp_mcqa)} ExtQA={len(fp_extqa)} AbsQA={len(fp_absqa)}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 5b: Build MinHashLSH indices over the fingerprints
# ──────────────────────────────────────────────────────────────────────
import time as _t
t0 = _t.time()

logger.info("Building MinHashLSH indices…")
lsh_mcqa,  _ = ec.build_lsh_index(fp_mcqa,  threshold=cfg.minhash_threshold,
                                  num_perm=cfg.minhash_num_perm)
lsh_extqa, _ = ec.build_lsh_index(fp_extqa, threshold=cfg.minhash_threshold,
                                  num_perm=cfg.minhash_num_perm)
lsh_absqa, _ = ec.build_lsh_index(fp_absqa, threshold=cfg.minhash_threshold,
                                  num_perm=cfg.minhash_num_perm)
logger.info(f"✓ Indices built in {ec.human_time(_t.time() - t0)}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 5c: Scan train + val rows; drop any near-duplicate of any test row
# ──────────────────────────────────────────────────────────────────────
from tqdm.auto import tqdm

def shield_filter(rows, fp_fn, lsh, label):
    kept, dropped = [], []
    for r in tqdm(rows, desc=f"  {label}", leave=False):
        fp = fp_fn(r)
        if fp and ec.is_leaked(fp, lsh, num_perm=cfg.minhash_num_perm):
            dropped.append(r)
        else:
            kept.append(r)
    pct = 100 * len(dropped) / max(len(rows), 1)
    logger.info(f"  {label}: kept {len(kept):,} / dropped {len(dropped):,} ({pct:.2f}% leakage)")
    return kept, dropped


# MCQA train + val against MCQA test
mcqa_train_clean,  mcqa_train_leaks  = shield_filter(mcqa_train_rows,
                                                      _mcqa_fingerprint, lsh_mcqa, "MCQA train")
mcqa_val_clean,    mcqa_val_leaks    = shield_filter(mcqa_val_rows,
                                                      _mcqa_fingerprint, lsh_mcqa, "MCQA val")

# ExtQA train + val against ExtQA test
extqa_train_clean, extqa_train_leaks = shield_filter(extqa_train_rows,
                                                      _extqa_fingerprint, lsh_extqa, "ExtQA train")
extqa_val_clean,   extqa_val_leaks   = shield_filter(extqa_val_rows,
                                                      _extqa_fingerprint, lsh_extqa, "ExtQA val")

# AbsQA train + val against AbsQA test
absqa_train_clean, absqa_train_leaks = shield_filter(absqa_train_rows,
                                                      _absqa_fingerprint, lsh_absqa, "AbsQA train")
absqa_val_clean,   absqa_val_leaks   = shield_filter(absqa_val_rows,
                                                      _absqa_fingerprint, lsh_absqa, "AbsQA val")

# Persist a leakage report — extremely useful for the paper's appendix
leakage_report = {
    "minhash_threshold": cfg.minhash_threshold,
    "minhash_num_perm":  cfg.minhash_num_perm,
    "mcqa": {
        "train_kept": len(mcqa_train_clean),
        "train_dropped": len(mcqa_train_leaks),
        "val_kept": len(mcqa_val_clean),
        "val_dropped": len(mcqa_val_leaks),
    },
    "extqa": {
        "train_kept": len(extqa_train_clean),
        "train_dropped": len(extqa_train_leaks),
        "val_kept": len(extqa_val_clean),
        "val_dropped": len(extqa_val_leaks),
    },
    "absqa": {
        "train_kept": len(absqa_train_clean),
        "train_dropped": len(absqa_train_leaks),
        "val_kept": len(absqa_val_clean),
        "val_dropped": len(absqa_val_leaks),
    },
}
report_path = ec.atomic_write_json(
    leakage_report, os.path.join(cfg.results_dir, "leakage_report.json"))
logger.info(f"✓ Leakage report → {report_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 6: Carve exemplar pools out of cleaned train sets
# ──────────────────────────────────────────────────────────────────────
import random as _r

def carve_exemplar_pool(rows, pool_size: int, seed: int = 42):
    rng = _r.Random(seed)
    if len(rows) <= pool_size:
        logger.warning(f"  pool size {pool_size} ≥ rows {len(rows)} — using all as pool!")
        return rows.copy(), []
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    pool_idx = set(idx[:pool_size])
    pool = [rows[i] for i in sorted(pool_idx)]
    train = [rows[i] for i in range(len(rows)) if i not in pool_idx]
    return pool, train


N_POOL = cfg.exemplar_pool_size_per_task

mcqa_pool,  mcqa_train_final  = carve_exemplar_pool(mcqa_train_clean,  N_POOL, cfg.seed)
extqa_pool, extqa_train_final = carve_exemplar_pool(extqa_train_clean, N_POOL, cfg.seed)
absqa_pool, absqa_train_final = carve_exemplar_pool(absqa_train_clean, N_POOL, cfg.seed)

logger.info(f"Exemplar pools (held out from training):")
logger.info(f"  MCQA  pool={len(mcqa_pool):>5} | train={len(mcqa_train_final):>6}")
logger.info(f"  ExtQA pool={len(extqa_pool):>5} | train={len(extqa_train_final):>6}")
logger.info(f"  AbsQA pool={len(absqa_pool):>5} | train={len(absqa_train_final):>6}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 7a: Schema unification helpers
# ──────────────────────────────────────────────────────────────────────
def normalise_mcqa(r, source="qanastek/frenchmedmcqa"):
    options = r.get("answers", {})
    correct = r.get("correct_answers", "")
    if isinstance(correct, list):
        correct = "".join(correct)
    correct = str(correct).upper().strip()
    # Letters only, sorted, comma-joined: "A,C"
    valid = sorted(set(c for c in correct if c.isalpha() and c.isupper()))
    return {
        "id":        str(r.get("id") or r.get("identifier") or ec.hash_text(r.get("question", ""))),
        "task":      "mcqa",
        "question":  r.get("question", ""),
        "context":   None,
        "options":   options if isinstance(options, dict) else None,
        "answer":    ",".join(valid),
        "answer_meta": {"raw_correct": str(correct)},
        "specialty": r.get("subject", r.get("specialty")),
        "source":    source,
    }


def normalise_extqa(r):
    a = r.get("answers", {})
    if isinstance(a, dict) and a.get("text"):
        ans = a["text"][0] if isinstance(a["text"], list) else str(a["text"])
        start = a.get("answer_start", [0])
        start = start[0] if isinstance(start, list) else start
    else:
        ans = str(a)
        start = 0
    return {
        "id":        str(r.get("id") or ec.hash_text(r.get("question", "") + ans)),
        "task":      "extqa",
        "question":  r.get("question", ""),
        "context":   r.get("context", ""),
        "options":   None,
        "answer":    ans,
        "answer_meta": {"answer_start": start},
        "specialty": r.get("specialty"),
        "source":    r.get("source", "QUAERO"),
    }


def normalise_absqa(r, source="ANR-MALADES/MediQAl"):
    # MediQAl variants: 'short_answer', 'long_answer', 'answer'
    ans = r.get("long_answer") or r.get("answer") or r.get("short_answer") or ""
    return {
        "id":        str(r.get("id") or r.get("question_id") or ec.hash_text(r.get("question", ""))),
        "task":      "absqa",
        "question":  r.get("question", ""),
        "context":   r.get("context"),
        "options":   None,
        "answer":    ans,
        "answer_meta": {
            "short_answer": r.get("short_answer"),
            "long_answer":  r.get("long_answer"),
        },
        "specialty": r.get("specialty") or r.get("speciality"),
        "source":    source,
    }

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 7b: Apply schema normalisation to all splits
# ──────────────────────────────────────────────────────────────────────
def _norm_all(rows, fn):
    return [fn(r) for r in rows]

train_rows = (
    _norm_all(mcqa_train_final,  normalise_mcqa) +
    _norm_all(extqa_train_final, normalise_extqa) +
    _norm_all(absqa_train_final, normalise_absqa)
)
val_rows = (
    _norm_all(mcqa_val_clean,  normalise_mcqa) +
    _norm_all(extqa_val_clean, normalise_extqa) +
    _norm_all(absqa_val_clean, normalise_absqa)
)
test_rows = (
    _norm_all(mcqa_test_rows,  normalise_mcqa) +
    _norm_all(extqa_test_rows, normalise_extqa) +
    _norm_all(absqa_test_rows, normalise_absqa)
)
exemplar_rows = (
    _norm_all(mcqa_pool,  normalise_mcqa) +
    _norm_all(extqa_pool, normalise_extqa) +
    _norm_all(absqa_pool, normalise_absqa)
)

# Shuffle train deterministically — mixes tasks so SFTTrainer sees variety per batch
import random as _r
rng = _r.Random(cfg.seed)
rng.shuffle(train_rows)

logger.info(f"Final unified rows:")
logger.info(f"  train     : {len(train_rows):,}")
logger.info(f"  val       : {len(val_rows):,}")
logger.info(f"  test      : {len(test_rows):,}")
logger.info(f"  exemplars : {len(exemplar_rows):,}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 7c: Persist JSONL artifacts (atomic writes)
# ──────────────────────────────────────────────────────────────────────
paths = {
    "train":     os.path.join(cfg.results_dir, "train.jsonl"),
    "val":       os.path.join(cfg.results_dir, "val.jsonl"),
    "test":      os.path.join(cfg.results_dir, "test.jsonl"),
    "exemplars": os.path.join(cfg.results_dir, "exemplars.jsonl"),
}

ec.atomic_write_jsonl(train_rows,    paths["train"])
ec.atomic_write_jsonl(val_rows,      paths["val"])
ec.atomic_write_jsonl(test_rows,     paths["test"])
ec.atomic_write_jsonl(exemplar_rows, paths["exemplars"])

# Also save the DAPT corpus pointer (NB2 will load full PARCOMED separately)
dapt_corpus_path = os.path.join(cfg.results_dir, "dapt_corpus_sample.jsonl")
ec.atomic_write_jsonl(
    [{"text": t} for t in (parcomed_texts + fr_medical_texts)[:5000]],
    dapt_corpus_path,
)

ec.drive_flush()

for name, path in paths.items():
    sz = os.path.getsize(path) / 1e6
    logger.info(f"  ✓ {name:9s} → {path}  ({sz:.1f} MB)")
logger.info(f"  ✓ DAPT sample → {dapt_corpus_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 8a: Compute summary statistics
# ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from collections import Counter

def summarise(rows, task_name):
    sub = [r for r in rows if r["task"] == task_name]
    if not sub:
        return None
    ans_lens = [len(str(r["answer"]).split()) for r in sub]
    specs = [r.get("specialty") for r in sub if r.get("specialty")]
    return {
        "task":               task_name,
        "n":                  len(sub),
        "answer_words_mean":  float(np.mean(ans_lens)) if ans_lens else 0.0,
        "answer_words_median": float(np.median(ans_lens)) if ans_lens else 0.0,
        "answer_words_p95":   float(np.percentile(ans_lens, 95)) if ans_lens else 0.0,
        "n_unique_specialties": len(set(specs)),
        "top5_specialties":   Counter(specs).most_common(5) if specs else [],
        "answer_distribution_top5": Counter(str(r["answer"])[:30] for r in sub).most_common(5),
    }


print("\n" + "=" * 78)
print(f"  EXPLORATORY ANALYSIS — split×task summary")
print("=" * 78)

stats = {}
for split_name, rows in [("train", train_rows), ("val", val_rows),
                         ("test", test_rows), ("exemplars", exemplar_rows)]:
    stats[split_name] = {}
    for task in ["mcqa", "extqa", "absqa"]:
        s = summarise(rows, task)
        if s:
            stats[split_name][task] = s

# Pretty print as a wide table
records = []
for split_name, by_task in stats.items():
    for task, s in by_task.items():
        records.append({
            "split":   split_name,
            "task":    task,
            "n":       s["n"],
            "ans_mean_words":   round(s["answer_words_mean"], 1),
            "ans_p95_words":    round(s["answer_words_p95"], 1),
            "n_specialties":    s["n_unique_specialties"],
        })
df_stats = pd.DataFrame(records).sort_values(["task", "split"])
print(df_stats.to_string(index=False))

ec.atomic_write_json(stats, os.path.join(cfg.results_dir, "data_stats.json"))

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 8b: Visualisations — answer-length histograms + specialty bars
# ──────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from collections import Counter

sns.set_theme(style="whitegrid", context="talk")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Data composition — train splits (post-shield)", fontsize=15, y=1.02)

for col, task in enumerate(["mcqa", "extqa", "absqa"]):
    sub = [r for r in train_rows if r["task"] == task]
    if not sub:
        axes[0, col].set_visible(False)
        axes[1, col].set_visible(False)
        continue
    # Top: answer length histogram
    lens = [len(str(r["answer"]).split()) for r in sub]
    axes[0, col].hist(lens, bins=30, color="#3a7", edgecolor="white")
    axes[0, col].set_title(f"{task.upper()} — answer length (words)")
    axes[0, col].set_xlabel("words")

    # Bottom: top specialties
    specs = [r.get("specialty") for r in sub if r.get("specialty")]
    if specs:
        top = Counter(specs).most_common(10)
        names, counts = zip(*top)
        axes[1, col].barh(range(len(names)), counts, color="#37c")
        axes[1, col].set_yticks(range(len(names)))
        axes[1, col].set_yticklabels(names, fontsize=10)
        axes[1, col].invert_yaxis()
        axes[1, col].set_title(f"{task.upper()} — top specialties")
    else:
        axes[1, col].set_visible(False)

plt.tight_layout()
fig_path = os.path.join(cfg.results_dir, "data_overview.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"✓ Overview figure → {fig_path}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 9: Hard validation gates
# ──────────────────────────────────────────────────────────────────────
REQUIRED_FIELDS = {"id", "task", "question", "answer", "source"}

def _validate(rows, label):
    for i, r in enumerate(rows):
        missing = REQUIRED_FIELDS - set(r.keys())
        assert not missing, f"{label}[{i}] missing fields: {missing}"
        assert r["task"] in {"mcqa", "extqa", "absqa"}, \
            f"{label}[{i}] bad task: {r['task']}"
        assert isinstance(r["question"], str) and r["question"], \
            f"{label}[{i}] empty question"

_validate(train_rows,    "train")
_validate(val_rows,      "val")
_validate(test_rows,     "test")
_validate(exemplar_rows, "exemplars")
logger.info("✓ Schema validation passed")

# Task-specific sanity checks
for r in test_rows:
    if r["task"] == "mcqa":
        assert r["answer"], f"MCQA test row {r['id']} has empty answer"
        assert all(c.isalpha() and c.isupper() for c in r["answer"].replace(",", "")), \
            f"MCQA test row {r['id']} has bad letters: {r['answer']}"
    elif r["task"] == "extqa":
        assert r["context"] and r["answer"], \
            f"ExtQA test row {r['id']} missing context or answer"
        # Span should appear in context (best-effort, not enforced strictly)
        if r["answer"] not in r["context"]:
            # Try whitespace-normalised match
            norm_ctx = " ".join(r["context"].split())
            norm_ans = " ".join(r["answer"].split())
            if norm_ans not in norm_ctx:
                logger.warning(f"  ExtQA span not literally in context: {r['id']}")
    elif r["task"] == "absqa":
        assert r["answer"], f"AbsQA test row {r['id']} has empty answer"
logger.info("✓ Task-specific sanity passed")

# Final cross-shield assertion: re-run shield on a sample of train against test indices,
# expecting zero leakage now.
import random as _random_check
_rng_check = _random_check.Random(cfg.seed)
sample_n = min(500, len(train_rows))
sample = _rng_check.sample(train_rows, sample_n) if sample_n else []

leaks_in_clean = 0
for r in sample:
    if r["task"] == "mcqa":
        fp = _mcqa_fingerprint({"question": r["question"], "answers": r["options"] or {}})
        if ec.is_leaked(fp, lsh_mcqa, num_perm=cfg.minhash_num_perm):
            leaks_in_clean += 1
    elif r["task"] == "extqa":
        fp = _extqa_fingerprint({"question": r["question"],
                                 "answers": {"text": [r["answer"]]}})
        if ec.is_leaked(fp, lsh_extqa, num_perm=cfg.minhash_num_perm):
            leaks_in_clean += 1
    elif r["task"] == "absqa":
        fp = _absqa_fingerprint({"question": r["question"]})
        if ec.is_leaked(fp, lsh_absqa, num_perm=cfg.minhash_num_perm):
            leaks_in_clean += 1
assert leaks_in_clean == 0, (
    f"✗ Shield failure: {leaks_in_clean}/{len(sample)} train rows still match test!"
)
logger.info(f"✓ Cross-shield: 0 leaks in {len(sample)}-row train sample")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 10: Upload data artifacts to HF Hub as a private dataset
# ──────────────────────────────────────────────────────────────────────
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi, create_repo

        repo_id = f"{HF_USER}/{cfg.project_name}-data"
        create_repo(repo_id, repo_type="dataset", private=True,
                    exist_ok=True, token=HF_TOKEN)
        api = HfApi(token=HF_TOKEN)
        for name, path in paths.items():
            api.upload_file(
                path_or_fileobj=path, path_in_repo=os.path.basename(path),
                repo_id=repo_id, repo_type="dataset",
            )
        # Plus the leakage report + stats
        for f in ["leakage_report.json", "data_stats.json", "data_overview.png"]:
            local = os.path.join(cfg.results_dir, f)
            if os.path.exists(local):
                api.upload_file(path_or_fileobj=local, path_in_repo=f,
                                repo_id=repo_id, repo_type="dataset")
        logger.info(f"✓ Pushed to https://huggingface.co/datasets/{repo_id}")
    except Exception as e:
        logger.error(f"HF upload failed (non-fatal): {e}")
else:
    logger.info("• Skipping HF upload (no token)")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Stage 11: Final summary banner
# ──────────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print(f"  NB1 COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  train.jsonl     : {len(train_rows):>7,} rows")
print(f"  val.jsonl       : {len(val_rows):>7,} rows")
print(f"  test.jsonl      : {len(test_rows):>7,} rows  ← locked Golden Test Set")
print(f"  exemplars.jsonl : {len(exemplar_rows):>7,} rows  (per-task pool)")
print(f"  Leakage filtered: {sum([len(mcqa_train_leaks), len(extqa_train_leaks), len(absqa_train_leaks)]):>7,} rows")
print("═" * 70)
print("  → Proceed to NB2_DAPT.ipynb for domain-adaptive pre-training.")
print("═" * 70)

## §3 — Domain-Adaptive Continual Pre-Training (DAPT)
*NB2 Stages 1–7 (your code verbatim)*

Loads Qwen3-14B in 4-bit from Drive cache, builds the French/English DAPT corpus (PARCOMED 90K + CAS + QUAERO + PubMed 95K + Wikipedia), applies the Qwen3 chat template, filters by `max_seq_length`, and trains a LoRA adapter with `packing=True`. Saves the adapter locally and pushes to `{HF_USER}/EnToFrMedicaLLM-DAPT`.

**Config:** LoRA r=32 α=64, lr=2e-5, 1 epoch, batch=2×grad_accum=8, max_seq=2048


In [ ]:
# Stage 1b — Config / logger / seed / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    cpt_lr=2e-5, cpt_epochs=1, cpt_batch_size=2, cpt_grad_accum=8,
    cpt_lora_r=32, cpt_lora_alpha=64, max_seq_length=2048,
)
cfg.make_dirs()
logger = ec.setup_logger("nb2",
    log_file=os.path.join(cfg.training_dir, f"nb2_dapt_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)
HF_USER = cfg.hf_user
USE_WANDB = ec.bootstrap_wandb(project=cfg.wandb_project, logger=logger)

DAPT_OUT_DIR = os.path.join(cfg.training_dir, "qwen3-14b-dapt")
os.makedirs(DAPT_OUT_DIR, exist_ok=True)
cfg.save()
logger.info("=" * 70)
logger.info(f"NB2 — DAPT  | base={cfg.base_model}")
logger.info(f"             | output={DAPT_OUT_DIR}")
logger.info(f"             | repo={cfg.hf_repo_dapt}")
logger.info("=" * 70)

In [ ]:
import torch, os
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

# Pin every HF cache env var to Drive BEFORE the loader runs. cfg.cache_dir
# is already on Drive (auto-derived to {base_dir}/hf_cache by enmed_core).
os.environ["HF_HOME"]                  = cfg.cache_dir
os.environ["TRANSFORMERS_CACHE"]       = cfg.cache_dir
os.environ["HUGGINGFACE_HUB_CACHE"]    = cfg.cache_dir
os.environ["HF_HUB_CACHE"]             = cfg.cache_dir
os.makedirs(cfg.cache_dir, exist_ok=True)

# Quick check: is the model already in the Drive cache?
from huggingface_hub import try_to_load_from_cache
hub_filename = "config.json"
cached_marker = try_to_load_from_cache(
    repo_id=cfg.base_model, filename=hub_filename, cache_dir=cfg.cache_dir,
)
if cached_marker:
    logger.info(f"▶ Loading from Drive HF cache (fast)")
else:
    logger.info(f"▶ First run — downloading {cfg.base_model} into Drive cache")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_length,
    load_in_4bit=cfg.load_in_4bit,
    dtype=None,
    token=HF_TOKEN,
    cache_dir=cfg.cache_dir,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
logger.info(f"  loaded | source={'Drive cache' if cached_marker else 'HF Hub'}"
            f" | params≈{sum(p.numel() for p in model.parameters())/1e9:.1f}B"
            f" | vocab={len(tokenizer):,}")

In [ ]:
# Stage 3a — FR medical: PARCOMED + CAS + QUAERO
from datasets import load_dataset
from tqdm.auto import tqdm
import random

DAPT_TARGET = {"fr_bio": 90_000, "en_bio": 95_000, "fr_gen": 10_000, "en_gen": 10_000}
fr_bio_texts, en_bio_texts, fr_gen_texts, en_gen_texts = [], [], [], []

def _take_text(ex):
    cands = [(k,v) for k,v in ex.items() if isinstance(v,str) and len(v) > 80]
    if not cands: return None
    cands.sort(key=lambda kv: -len(kv[1])); return cands[0][1]

# PARCOMED — primary
try:
    pc = load_dataset(cfg.parcomed_id, split="train", streaming=True,
                      cache_dir=cfg.cache_dir, token=HF_TOKEN)
    for ex in tqdm(pc, desc="PARCOMED"):
        if len(fr_bio_texts) >= DAPT_TARGET["fr_bio"]: break
        t = _take_text(ex)
        if t: fr_bio_texts.append(t[:3000])
    logger.info(f"  PARCOMED: {len(fr_bio_texts):,}")
except Exception as e:
    logger.warning(f"  PARCOMED unavailable: {e}")

# CAS
for ds_id in ["DrBenchmark/CAS", "Dr-BERT/CAS"]:
    if len(fr_bio_texts) >= DAPT_TARGET["fr_bio"]: break
    try:
        cas = load_dataset(ds_id, cache_dir=cfg.cache_dir,
                           token=HF_TOKEN, trust_remote_code=True)
        for split in cas.keys():
            for ex in cas[split]:
                if len(fr_bio_texts) >= DAPT_TARGET["fr_bio"]: break
                if "tokens" in ex and ex["tokens"]:
                    t = " ".join(str(x) for x in ex["tokens"])
                else:
                    t = _take_text(ex) or ""
                if len(t) > 100: fr_bio_texts.append(t[:3000])
        logger.info(f"  CAS {ds_id}: cum FR bio = {len(fr_bio_texts):,}")
        break
    except Exception as e:
        logger.warning(f"  {ds_id} failed: {e}")

# QUAERO
for subset in ["medline", "emea"]:
    if len(fr_bio_texts) >= DAPT_TARGET["fr_bio"]: break
    try:
        qd = load_dataset("Dr-BERT/QUAERO", subset, cache_dir=cfg.cache_dir,
                          token=HF_TOKEN, trust_remote_code=True)
        for split in qd.keys():
            for ex in qd[split]:
                if len(fr_bio_texts) >= DAPT_TARGET["fr_bio"]: break
                if "tokens" in ex and ex["tokens"]:
                    fr_bio_texts.append(" ".join(str(x) for x in ex["tokens"])[:3000])
        logger.info(f"  QUAERO/{subset}: cum = {len(fr_bio_texts):,}")
    except Exception as e:
        logger.warning(f"  QUAERO/{subset}: {e}")

logger.info(f"✓ FR bio total: {len(fr_bio_texts):,}")

In [ ]:
# Stage 3b — EN PubMed (replay) + FR/EN Wikipedia (general replay)
try:
    pm = load_dataset("ccdv/pubmed-summarization", "document",
                      split=f"train[:{DAPT_TARGET['en_bio']}]",
                      cache_dir=cfg.cache_dir)
    col = next((c for c in pm.column_names
                if any(k in c.lower() for k in ["abstract","article","text"])),
               pm.column_names[0])
    for ex in pm:
        t = str(ex.get(col,"")).strip()
        if t: en_bio_texts.append(t[:3000])
    logger.info(f"  PubMed: {len(en_bio_texts):,}")
except Exception as e:
    logger.warning(f"  PubMed: {e}")

try:
    wf = load_dataset("wikimedia/wikipedia", "20231101.fr",
                      split=f"train[:{DAPT_TARGET['fr_gen']}]",
                      cache_dir=cfg.cache_dir)
    for ex in wf:
        t = (str(ex.get("title","")) + "\n" + str(ex.get("text",""))[:2500]).strip()
        if t: fr_gen_texts.append(t)
    logger.info(f"  FR wiki: {len(fr_gen_texts):,}")
except Exception as e:
    logger.warning(f"  FR wiki: {e}")

try:
    we = load_dataset("wikimedia/wikipedia", "20231101.en",
                      split=f"train[:{DAPT_TARGET['en_gen']}]",
                      cache_dir=cfg.cache_dir)
    for ex in we:
        t = (str(ex.get("title","")) + "\n" + str(ex.get("text",""))[:2500]).strip()
        if t: en_gen_texts.append(t)
    logger.info(f"  EN wiki: {len(en_gen_texts):,}")
except Exception as e:
    logger.warning(f"  EN wiki: {e}")

logger.info(f"✓ Pools | FR_bio={len(fr_bio_texts):,}  EN_bio={len(en_bio_texts):,}  "
            f"FR_gen={len(fr_gen_texts):,}  EN_gen={len(en_gen_texts):,}")

In [ ]:
# Stage 3c — Convert texts → conversations → HF dataset (with chat template)
import re
from datasets import Dataset as HFDataset
import multiprocessing as mp

def _to_conv(text, lang):
    text = (text or "").strip()
    if len(text) < 50: return None
    if len(text) > 3000: text = text[:3000]
    if lang == "fr":
        u = "Résume et explique les points clés de ce texte médical en français."
        ctx_lbl = "Contexte médical"
    else:
        u = "Summarize and explain the key points of this medical text."
        ctx_lbl = "Medical context"
    return [{"role": "user",      "content": f"{u}\n\n{ctx_lbl}:\n{text}"},
            {"role": "assistant", "content": text}]

def _build(texts, lang, label, target):
    rng = random.Random(cfg.seed)
    if len(texts) > target: texts = rng.sample(texts, target)
    convos = [c for c in (_to_conv(t, lang)
              for t in tqdm(texts, desc=f"  {label}", leave=False)) if c]
    logger.info(f"  {label}: {len(convos):,} conversations")
    return convos

all_convos = []
all_convos += _build(fr_bio_texts, "fr", "FR bio",  DAPT_TARGET["fr_bio"])
all_convos += _build(en_bio_texts, "en", "EN bio",  DAPT_TARGET["en_bio"])
all_convos += _build(fr_gen_texts, "fr", "FR gen",  DAPT_TARGET["fr_gen"])
all_convos += _build(en_gen_texts, "en", "EN gen",  DAPT_TARGET["en_gen"])
random.Random(cfg.seed).shuffle(all_convos)

dapt_ds = HFDataset.from_dict({"conversations": all_convos})
del all_convos, fr_bio_texts, en_bio_texts, fr_gen_texts, en_gen_texts
ec.cleanup()
logger.info(f"✓ Raw: {len(dapt_ds):,}")

In [ ]:
# Stage 3d — Apply template, length filter, train/val split
NUM_PROC = max(1, mp.cpu_count() - 1)

def _apply(ex):
    out = []
    for c in ex["conversations"]:
        try:
            out.append(tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False))
        except Exception:
            out.append(None)
    return {"text": out}

dapt_ds = dapt_ds.map(_apply, batched=True, num_proc=NUM_PROC)
dapt_ds = dapt_ds.filter(lambda x: x["text"] is not None and len(x["text"]) > 0)

_tok = getattr(tokenizer, "tokenizer", tokenizer)
def _len_ok(batch):
    ids = _tok(batch["text"], truncation=False, padding=False,
               add_special_tokens=False)["input_ids"]
    return [len(x) <= cfg.max_seq_length for x in ids]
before = len(dapt_ds)
dapt_ds = dapt_ds.filter(_len_ok, batched=True, num_proc=NUM_PROC)
logger.info(f"  Length-filter: {len(dapt_ds):,} (dropped {before - len(dapt_ds):,})")

split = dapt_ds.train_test_split(test_size=0.05, seed=cfg.seed)
train_ds, val_ds = split["train"], split["test"]
logger.info(f"✓ DAPT splits | train={len(train_ds):,}  val={len(val_ds):,}")

In [ ]:
# Stage 4a — LoRA adapter
model = FastLanguageModel.get_peft_model(
    model, r=cfg.cpt_lora_r, lora_alpha=cfg.cpt_lora_alpha, lora_dropout=0.0,
    target_modules=cfg.sft_lora_targets, bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg.seed, use_rslora=False, loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
# Stage 4b — SFTTrainer with packing=True
import math
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

total_steps = math.ceil(len(train_ds) / (cfg.cpt_batch_size * cfg.cpt_grad_accum)) * cfg.cpt_epochs
warmup = max(1, int(total_steps * cfg.cpt_warmup_ratio))
save_steps = max(1, int(total_steps * cfg.save_every_n_steps_frac))
logger.info(f"DAPT plan | steps={total_steps:,}  warmup={warmup}  save_every={save_steps}")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=DAPT_OUT_DIR,
        num_train_epochs=cfg.cpt_epochs,
        per_device_train_batch_size=cfg.cpt_batch_size,
        gradient_accumulation_steps=cfg.cpt_grad_accum,
        learning_rate=cfg.cpt_lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=max(1, total_steps // 100),
        save_strategy="steps", save_steps=save_steps,
        save_total_limit=cfg.keep_n_checkpoints,
        eval_strategy="steps", eval_steps=save_steps,
        report_to="wandb" if USE_WANDB else "none",
        run_name=f"DAPT_Qwen3-14B_{ec.now_ts()}",
        seed=cfg.seed, gradient_checkpointing=True,
        optim="adamw_8bit", weight_decay=0.01, max_grad_norm=1.0,
        max_seq_length=cfg.max_seq_length, dataset_text_field="text",
        packing=True, push_to_hub=False,
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n")

logger.info("▶ Starting DAPT…")
result = ec.resumable_train(trainer, DAPT_OUT_DIR,
                            auto_resume=cfg.auto_resume, logger=logger)
ppl_train = math.exp(result.training_loss) if result.training_loss < 20 else float("inf")
logger.info(f"✓ DAPT done | train_loss={result.training_loss:.4f}  ppl≈{ppl_train:.2f}")

model.save_pretrained(DAPT_OUT_DIR)
tokenizer.save_pretrained(DAPT_OUT_DIR)
ec.drive_flush()

In [ ]:
# Stage 5 — Final perplexity on held-out val
import math
ev = trainer.evaluate(eval_dataset=val_ds)
val_loss = ev.get("eval_loss", float("nan"))
val_ppl  = math.exp(val_loss) if val_loss < 20 else float("inf")
logger.info(f"  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}")

ec.atomic_write_json({
    "train_loss": float(result.training_loss),
    "train_ppl":  float(ppl_train),
    "val_loss":   float(val_loss),
    "val_ppl":    float(val_ppl),
    "total_steps": int(total_steps),
    "config": {"r": cfg.cpt_lora_r, "alpha": cfg.cpt_lora_alpha,
               "lr": cfg.cpt_lr, "max_seq": cfg.max_seq_length,
               "batch": cfg.cpt_batch_size, "grad_accum": cfg.cpt_grad_accum,
               "packing": True},
}, os.path.join(cfg.results_dir, "dapt_metrics.json"))

In [ ]:
# Stage 6 — Upload to {HF_USER}/EnToFrMedicaLLM-DAPT
if HF_TOKEN:
    try:
        model.push_to_hub(cfg.hf_repo_dapt, token=HF_TOKEN)
        tokenizer.push_to_hub(cfg.hf_repo_dapt, token=HF_TOKEN)
        logger.info(f"✓ DAPT adapter → https://huggingface.co/{cfg.hf_repo_dapt}")
    except Exception as e:
        logger.error(f"HF push failed: {e}")
else:
    logger.warning("• Skipping HF push (no token)")

In [ ]:
print("═" * 70)
print(f"  NB2 (DAPT) COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  Local : {DAPT_OUT_DIR}")
print(f"  HF    : {cfg.hf_repo_dapt}")
print(f"  Train : loss={result.training_loss:.4f}  ppl≈{ppl_train:.2f}")
print(f"  Val   : loss={val_loss:.4f}  ppl={val_ppl:.2f}")
print("═" * 70)
print("  → Open NB3_SFT.ipynb")

## §4 — Multi-Task Supervised Fine-Tuning (SFT)
*NB3 Stages 1–6 (your code verbatim)*

Loads NB1 train/val splits, loads the DAPT-merged backbone (base + merged DAPT adapter), and trains **four LoRA adapters** sequentially:
- **EnMed-MCQA** (3 epochs) — MCQA only
- **EnMed-ExtQA** (3 epochs) — ExtQA only
- **EnMed-AbsQA** (2 epochs) — AbsQA only
- **EnMed-Unified** (2 epochs) — all three tasks mixed → **headline model**

Each adapter is pushed to HF Hub after training.

**Config:** LoRA r=32 α=64 dropout=0.05, lr=2e-4, batch=2×grad_accum=8, adamw_8bit cosine


In [ ]:
# Stage 1b — Config / logger / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    sft_lr=2e-4, sft_lora_r=32, sft_lora_alpha=64, sft_lora_dropout=0.05,
    sft_epochs_mcqa=3, sft_epochs_extqa=3, sft_epochs_absqa=2, sft_epochs_unified=2,
    sft_batch_size=2, sft_grad_accum=8, sft_seeds=[42],
    max_seq_length=2048,
)
cfg.make_dirs()
logger = ec.setup_logger("nb3",
    log_file=os.path.join(cfg.training_dir, f"nb3_sft_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)
USE_WANDB = ec.bootstrap_wandb(project=cfg.wandb_project, logger=logger)

# Adapter output paths (one per task)
SFT_DIRS = {
    "mcqa":    os.path.join(cfg.training_dir, "qwen3-14b-sft-mcqa"),
    "extqa":   os.path.join(cfg.training_dir, "qwen3-14b-sft-extqa"),
    "absqa":   os.path.join(cfg.training_dir, "qwen3-14b-sft-absqa"),
    "unified": os.path.join(cfg.training_dir, "qwen3-14b-sft-unified"),
}
for d in SFT_DIRS.values(): os.makedirs(d, exist_ok=True)

HF_REPOS = {
    "mcqa":    f"{cfg.hf_user}/{cfg.project_name}-MCQA",
    "extqa":   f"{cfg.hf_user}/{cfg.project_name}-ExtQA",
    "absqa":   f"{cfg.hf_user}/{cfg.project_name}-AbsQA",
    "unified": cfg.hf_repo_unified,
}
cfg.save()
logger.info("=" * 70)
logger.info(f"NB3 — SFT  | DAPT base: {cfg.hf_repo_dapt}")
for k, d in SFT_DIRS.items():
    logger.info(f"           | {k:8s} → {HF_REPOS[k]}")
logger.info("=" * 70)

In [ ]:
# Stage 2a — Load train/val from NB1
TRAIN_PATH = os.path.join(cfg.results_dir, "train.jsonl")
VAL_PATH   = os.path.join(cfg.results_dir, "val.jsonl")
for p in (TRAIN_PATH, VAL_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

train_rows = ec.load_jsonl(TRAIN_PATH)
val_rows   = ec.load_jsonl(VAL_PATH)
logger.info(f"  train={len(train_rows):,}  val={len(val_rows):,}")

# Per-task partitions
def _by_task(rows, task): return [r for r in rows if r["task"] == task]
TASK_TRAIN = {t: _by_task(train_rows, t) for t in ["mcqa","extqa","absqa"]}
TASK_VAL   = {t: _by_task(val_rows,   t) for t in ["mcqa","extqa","absqa"]}
for t in ["mcqa","extqa","absqa"]:
    logger.info(f"  {t:6s} | train={len(TASK_TRAIN[t]):,}  val={len(TASK_VAL[t]):,}")

In [ ]:
# Stage 2b — Convert each row → Qwen3-thinking conversation
def _opts_dict_or_list(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}

def row_to_messages(r):
    """Build the supervised conversation for one row."""
    if r["task"] == "mcqa":
        msgs = ec.build_mcqa_messages(r["question"], _opts_dict_or_list(r["options"]))
    elif r["task"] == "extqa":
        msgs = ec.build_extqa_messages(r.get("context",""), r["question"])
    else:  # absqa
        msgs = ec.build_absqa_messages(r["question"], r.get("context"))
    msgs.append({"role": "assistant",
                 "content": r["answer"]})
    return msgs

In [ ]:
# Stage 3a — Factory: load DAPT base + attach a fresh LoRA + return trainer
import math, torch
from datasets import Dataset as HFDataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
import multiprocessing as mp
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
NUM_PROC = max(1, mp.cpu_count() - 1)

def build_train_pack(rows):
    """Rows → templated text dataset filtered to ≤ max_seq_length."""
    convos = [row_to_messages(r) for r in rows]
    ds = HFDataset.from_dict({"conversations": convos})
    def _apply(ex):
        out = []
        for c in ex["conversations"]:
            try:
                out.append(tokenizer.apply_chat_template(
                    c, tokenize=False, add_generation_prompt=False))
            except Exception:
                out.append(None)
        return {"text": out}
    ds = ds.map(_apply, batched=True, num_proc=NUM_PROC)
    ds = ds.filter(lambda x: x["text"] is not None and len(x["text"]) > 0)
    _tok = getattr(tokenizer, "tokenizer", tokenizer)
    def _len_ok(b):
        ids = _tok(b["text"], truncation=False, padding=False,
                   add_special_tokens=False)["input_ids"]
        return [len(x) <= cfg.max_seq_length for x in ids]
    ds = ds.filter(_len_ok, batched=True, num_proc=NUM_PROC)
    return ds


def make_trainer(train_ds, val_ds, output_dir, run_name, n_epochs, seed):
    total_steps = math.ceil(
        len(train_ds) / (cfg.sft_batch_size * cfg.sft_grad_accum)) * n_epochs
    warmup = max(1, int(total_steps * 0.05))
    save_steps = max(1, int(total_steps * cfg.save_every_n_steps_frac))

    args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=n_epochs,
        per_device_train_batch_size=cfg.sft_batch_size,
        gradient_accumulation_steps=cfg.sft_grad_accum,
        learning_rate=cfg.sft_lr,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=max(1, total_steps // 100),
        save_strategy="steps", save_steps=save_steps,
        save_total_limit=cfg.keep_n_checkpoints,
        eval_strategy="steps", eval_steps=save_steps,
        report_to="wandb" if USE_WANDB else "none",
        run_name=run_name,
        seed=seed, gradient_checkpointing=True,
        optim="adamw_8bit", weight_decay=0.01, max_grad_norm=1.0,
        max_seq_length=cfg.max_seq_length,
        dataset_text_field="text",
        packing=True, push_to_hub=False,
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds, args=args,
    )
    return train_on_responses_only(trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n")

In [ ]:
# Stage 3b — Load BASE in 4-bit, overlay DAPT adapter, then merge → clean base.
#
# Why this two-step load instead of `from_pretrained(cfg.hf_repo_dapt)`:
#   - Loading a LoRA-on-4bit repo directly via Unsloth tries to fit base+adapter
#     in VRAM with a bad device map → `ValueError: dispatched on CPU/disk` on L4.
#   - Loading the original 4-bit base first (~8 GB) then overlaying the LoRA
#     (~200 MB) and merging gives us a clean merged base that:
#       (a) fits the L4's 22 GB easily, and
#       (b) has no PEFT wrapper, so `attach_fresh_lora` in Stage 4 can
#           wrap a brand-new adapter without "model already has LoRA" errors.
from peft import PeftModel

logger.info(f"Loading base in 4-bit: {cfg.base_model}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_length,
    load_in_4bit=cfg.load_in_4bit,
    dtype=None, token=HF_TOKEN, cache_dir=cfg.cache_dir,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
logger.info(f"  base loaded | vocab={len(tokenizer):,}")

logger.info(f"Overlaying DAPT adapter: {cfg.hf_repo_dapt}")
model = PeftModel.from_pretrained(model, cfg.hf_repo_dapt, token=HF_TOKEN)

logger.info("  Merging DAPT adapter into base weights…")
model = model.merge_and_unload()
logger.info(f"  ✓ DAPT merged | params≈{sum(p.numel() for p in model.parameters())/1e9:.1f}B")

In [ ]:
# Stage 4 — Train each task adapter; multi-seed if cfg.sft_seeds has >1 entry
def attach_fresh_lora(seed):
    """Wrap the (clean) base model with a brand-new task LoRA.

    Guard: if a previous task left a LoRA on the model, merge_and_unload it
    first so Unsloth doesn't refuse with "model already has LoRA adapters".
    """
    global model
    if hasattr(model, "merge_and_unload"):
        try:
            model = model.merge_and_unload()
        except Exception as _e:
            logger.warning(f"  merge_and_unload skipped: {_e}")
    return FastLanguageModel.get_peft_model(
        model,
        r=cfg.sft_lora_r, lora_alpha=cfg.sft_lora_alpha,
        lora_dropout=cfg.sft_lora_dropout,
        target_modules=cfg.sft_lora_targets, bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=seed, use_rslora=False, loftq_config=None,
    )


def train_one(task, train_rows, val_rows, n_epochs):
    """Train one adapter (one seed) on a subset of rows."""
    global model
    seed = cfg.sft_seeds[0]
    out_dir = SFT_DIRS[task]
    run = f"SFT_{task}_seed{seed}_{ec.now_ts()}"
    logger.info(f"=== {task.upper()} | rows={len(train_rows):,} | epochs={n_epochs} | seed={seed} ===")

    # Strip any prior task adapter, then attach a fresh one
    model = attach_fresh_lora(seed)
    model.print_trainable_parameters()

    train_ds = build_train_pack(train_rows)
    val_ds   = build_train_pack(val_rows)
    trainer = make_trainer(train_ds, val_ds, out_dir, run, n_epochs, seed)

    result = ec.resumable_train(trainer, out_dir, auto_resume=cfg.auto_resume, logger=logger)
    logger.info(f"  {task} | train_loss={result.training_loss:.4f}")

    # Save + push
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    ec.drive_flush()

    if HF_TOKEN:
        try:
            model.push_to_hub(HF_REPOS[task], token=HF_TOKEN)
            tokenizer.push_to_hub(HF_REPOS[task], token=HF_TOKEN)
            logger.info(f"  ✓ pushed → https://huggingface.co/{HF_REPOS[task]}")
        except Exception as e:
            logger.error(f"  push failed: {e}")

    metrics = {
        "task": task, "seed": int(seed), "n_epochs": int(n_epochs),
        "train_loss": float(result.training_loss),
        "n_train": len(train_rows), "n_val": len(val_rows),
    }
    ec.atomic_write_json(metrics, os.path.join(cfg.results_dir, f"sft_{task}_metrics.json"))
    return metrics

In [ ]:
# Stage 4a — MCQA adapter
m_mcqa = train_one("mcqa", TASK_TRAIN["mcqa"], TASK_VAL["mcqa"],
                   n_epochs=cfg.sft_epochs_mcqa)
ec.cleanup()

In [ ]:
# Stage 4b — ExtQA adapter
m_extqa = train_one("extqa", TASK_TRAIN["extqa"], TASK_VAL["extqa"],
                    n_epochs=cfg.sft_epochs_extqa)
ec.cleanup()

In [ ]:
# Stage 4c — AbsQA adapter
m_absqa = train_one("absqa", TASK_TRAIN["absqa"], TASK_VAL["absqa"],
                    n_epochs=cfg.sft_epochs_absqa)
ec.cleanup()

In [ ]:
# Stage 4d — Unified adapter (all three tasks mixed) — the headline model
import random as _r
unified_train = TASK_TRAIN["mcqa"] + TASK_TRAIN["extqa"] + TASK_TRAIN["absqa"]
unified_val   = TASK_VAL["mcqa"]   + TASK_VAL["extqa"]   + TASK_VAL["absqa"]
_r.Random(cfg.seed).shuffle(unified_train)
m_unif = train_one("unified", unified_train, unified_val,
                   n_epochs=cfg.sft_epochs_unified)
ec.cleanup()

In [ ]:
# Stage 5 — Compact loss table
import pandas as pd
df = pd.DataFrame([m_mcqa, m_extqa, m_absqa, m_unif])
print(df[["task","n_train","n_val","n_epochs","train_loss"]].to_string(index=False))
ec.atomic_write_json(df.to_dict(orient="records"),
                     os.path.join(cfg.results_dir, "sft_summary.json"))

In [ ]:
print("═" * 70)
print(f"  NB3 (SFT) COMPLETE — {ec.now_ts()}")
print("═" * 70)
for m in [m_mcqa, m_extqa, m_absqa, m_unif]:
    print(f"  {m['task']:8s} loss={m['train_loss']:.4f}  →  {HF_REPOS[m['task']]}")
print("═" * 70)
print("  → Open NB4_ExportQuant.ipynb")

## §5 — Baseline Evaluation (Vanilla Models)
*NB5a Stages 1–7 (your code verbatim)*

Evaluates the three vanilla baselines (Qwen3-14B-vanilla, Qwen3-8B, Mistral-7B-Instruct-v0.3) across all 9 (task × shot) cells. Uses the `UnslothEngine` with crash-safe JSON caching — already-evaluated cells are skipped on re-run.

**Runners:**
- `run_mcqa` — letter-set prediction, strips `<think>` blocks
- `run_extqa` — verbatim span extraction, first-line trimming
- `run_absqa` — free-form generation, `_clean_gen` post-processing


In [ ]:
# Stage 1b — Config / logger / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    eval_n_shots=[0, 3, 5],
    eval_max_new_tokens_mcqa=64,     # thinking suppressed; 64 is generous for letters
    eval_max_new_tokens_extqa=64,    # cut hard: extracted spans must be SHORT
    eval_max_new_tokens_absqa=512,
    eval_temperature=0.0,
    eval_subset_size=None,   # None = full test set; set int for debug
)
cfg.make_dirs()
logger = ec.setup_logger("nb5",
    log_file=os.path.join(cfg.evals_dir, f"nb5_eval_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)

EVAL_CACHE_DIR = os.path.join(cfg.evals_dir, "results_cache")
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
logger.info(f"Eval cache dir: {EVAL_CACHE_DIR}")

In [ ]:
# Stage 2 — Read NB1 outputs
TEST_PATH = os.path.join(cfg.results_dir, "test.jsonl")
EXEM_PATH = os.path.join(cfg.results_dir, "exemplars.jsonl")
for p in (TEST_PATH, EXEM_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

test_rows  = ec.load_jsonl(TEST_PATH)
exem_rows  = ec.load_jsonl(EXEM_PATH)

if cfg.eval_subset_size:
    import random as _r
    _r.Random(cfg.seed).shuffle(test_rows)
    test_rows = test_rows[:cfg.eval_subset_size]
    logger.warning(f"DEBUG MODE — capped test to {len(test_rows)}")

def _by_task(rows, t): return [r for r in rows if r["task"] == t]
TEST = {t: _by_task(test_rows, t) for t in ["mcqa","extqa","absqa"]}
POOL = {t: _by_task(exem_rows, t) for t in ["mcqa","extqa","absqa"]}
logger.info(f"  TEST  | mcqa={len(TEST['mcqa']):,}  extqa={len(TEST['extqa']):,}  absqa={len(TEST['absqa']):,}")
logger.info(f"  POOL  | mcqa={len(POOL['mcqa']):,}  extqa={len(POOL['extqa']):,}  absqa={len(POOL['absqa']):,}")

In [ ]:
# Stage 3 — Model registry: vanilla baselines only.
#
# Each entry: {name, base, adapters}.
# For NB5_a we evaluate ONLY the four vanilla baseline models. The EnMed
# adapter chain models live in NB5_b. Both notebooks write into the same
# EVAL_CACHE_DIR so the final summary table aggregates everything.

REGISTRY = [
    {"name": "Qwen3-14B-vanilla",        "base": "unsloth/Qwen3-14B-unsloth-bnb-4bit",        "adapters": []},
    {"name": "Qwen3-8B",                 "base": "unsloth/Qwen3-8B-unsloth-bnb-4bit",         "adapters": []},
    {"name": "Mistral-7B-Instruct-v0.3", "base": "unsloth/mistral-7b-instruct-v0.3-bnb-4bit", "adapters": []},
    {"name": "Llama-2-13B",              "base": "unsloth/llama-2-13b-bnb-4bit",              "adapters": []},
]
for e in REGISTRY:
    chain = " → ".join([e["base"]] + e["adapters"])
    logger.info(f"  {e['name']:30s}  {chain}")

In [ ]:
# Stage 4 — Inference engine. Loads base in 4-bit, then overlays adapters in
# order via PEFT, merging each before the next so we end with a single merged
# 4-bit model ready for fast inference.
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
import torch, warnings, gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)


def _hard_vram_reset():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    gc.collect()


class UnslothEngine:
    def __init__(self, base, adapters=None, max_seq=2048):
        from peft import PeftModel
        _hard_vram_reset()

        self.base = base
        self.adapters = list(adapters or [])

        # Pin everything to GPU 0. device_map="auto" lets Accelerate split the
        # model across CPU/GPU which then trips the bnb-4bit validator with
        # "dispatched on CPU/disk" — even when the model would fit in VRAM.
        load_kwargs = dict(
            model_name=base, max_seq_length=max_seq,
            load_in_4bit=True, dtype=None,
            token=HF_TOKEN, cache_dir=cfg.cache_dir,
        )
        try:
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                **load_kwargs, device_map={"": 0},
            )
        except TypeError:
            # Older Unsloth signature — falls back to auto with explicit max_memory
            free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 22
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                **load_kwargs,
                max_memory={0: f"{int(free_gib - 1)}GiB"},
            )

        # Apply adapters in order, merging each before the next
        for i, adapter_repo in enumerate(self.adapters, 1):
            logger.info(f"    overlaying adapter [{i}/{len(self.adapters)}] {adapter_repo}")
            self.model = PeftModel.from_pretrained(
                self.model, adapter_repo, token=HF_TOKEN)
            self.model = self.model.merge_and_unload()
            _hard_vram_reset()

        try:
            self.tokenizer = get_chat_template(self.tokenizer,
                                               chat_template="qwen3")
        except Exception:
            pass
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        FastLanguageModel.for_inference(self.model)

        # Strip max_length from default GenerationConfig — avoids the
        # "Both max_new_tokens and max_length set" warning spam.
        if hasattr(self.model, "generation_config") and self.model.generation_config is not None:
            self.model.generation_config.max_length = None
            self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
            if self.model.generation_config.eos_token_id is None:
                self.model.generation_config.eos_token_id = self.tokenizer.eos_token_id

    @torch.inference_mode()
    def generate(self, messages, max_new_tokens=256, temperature=0.0):
        """Generate with thinking SUPPRESSED at three levels.

        Qwen3 was pretrained with thinking enabled, so even with the plain
        `qwen3` chat template it tends to emit a <think>...</think> block
        before the answer. With small token budgets (e.g. MCQA=32) the
        model exhausts its budget mid-thinking and `pred` ends up empty.

        We force the model to skip thinking via three layers of defense:
          1. Pass `enable_thinking=False` to apply_chat_template — Qwen3's
             tokenizer recognizes this kwarg and inserts an empty thinking
             block in the prompt header, signaling "thinking already done".
          2. Append literal "<think></think>\n\n" to the prompt as a
             mechanical fallback for tokenizers that ignore (1).
          3. Strip any residual <think>...</think> from the output as a
             final safety net (defense in depth).
        """
        # Layer 1 + 2: build prompt with thinking suppressed
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,            # ← Qwen3 specific
            )
        except TypeError:
            # Older tokenizer doesn't accept enable_thinking — fall back
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = "\n".join(f"{m['role']}: {m['content']}"
                                for m in messages) + "\nassistant:"

        # Layer 2 (mechanical fallback): if the prompt doesn't already end
        # with a closed think block, append one. Qwen3 will then continue
        # generating *after* it, producing the answer directly.
        if "<think></think>" not in prompt and "<think>\n\n</think>" not in prompt:
            prompt = prompt.rstrip() + "<think></think>\n\n"

        inputs = self.tokenizer(prompt, return_tensors="pt",
                                return_attention_mask=True).to(self.model.device)
        out = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            max_length=None,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9 if temperature > 0 else 1.0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(gen, skip_special_tokens=True)

        # Layer 3 (safety net): strip any <think>...</think> that leaked
        # through, plus dangling <think> from truncated outputs.
        import re as _re
        text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL)
        text = _re.sub(r"<think>.*",          "", text, flags=_re.DOTALL)
        return text.strip()

    def close(self):
        # Important: explicit del + reset, not just `del self.model`,
        # because PEFT/Unsloth keep weak references that hold VRAM.
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.tokenizer
        _hard_vram_reset()

In [ ]:
# Stage 5a — MCQA runner
import re
from tqdm.auto import tqdm

# Local cleaning helpers (kept here so this cell is self-contained for runners).
_RUN_THINK_RE       = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_RUN_DANGLING_THINK = re.compile(r"<think>.*",          flags=re.DOTALL | re.IGNORECASE)
_RUN_TAG_RE         = re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                                 flags=re.IGNORECASE)
_RUN_IM_TOKEN       = re.compile(r"<\|im_(?:start|end)\|>(?:assistant|user|system)?")

def _clean_gen(text):
    """Strip <think>, dangling <think>, and chat control tokens."""
    if not text: return ""
    s = _RUN_THINK_RE.sub("", text)
    s = _RUN_DANGLING_THINK.sub("", s)
    s = _RUN_TAG_RE.sub("", s)
    s = _RUN_IM_TOKEN.sub("", s)
    return s.strip()


def _opts(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}


def run_mcqa(engine, test, pool, n_shot, model_name, seed=42):
    out_rows = []
    for ex in tqdm(test, desc=f"  MCQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"question": d["question"],
                  "options":  _opts(d["options"]),
                  "answer":   d["answer"]} for d in demos_raw]
        msgs = ec.build_mcqa_messages(ex["question"], _opts(ex["options"]), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_mcqa,
                              temperature=cfg.eval_temperature)
        # Strip <think> from raw before parsing letters
        cleaned = _clean_gen(gen)
        n_choices = len(_opts(ex["options"])) or 5
        pred_letters = ec.parse_mcqa_letters(cleaned, n_choices=n_choices)
        gold_letters = sorted(set(c for c in (ex["answer"] or "").split(",")
                                  if c.strip().isalpha()))
        out_rows.append({
            "id": ex["id"], "n_shot": n_shot,
            "raw":  gen, "cleaned": cleaned,
            "pred": ",".join(pred_letters),
            "gold": ",".join(gold_letters),
            "n_choices": n_choices,
        })
    return out_rows


def score_mcqa(rows):
    """Letter-set accuracy + macro F1 + Hamming over A..Z."""
    import numpy as np
    if not rows: return {}
    set_acc = np.mean([set(r["pred"].split(",")) == set(r["gold"].split(","))
                       for r in rows])
    n_choices = max(r["n_choices"] for r in rows)
    f1s = []
    for i in range(n_choices):
        L = chr(65+i)
        yt = [1 if L in r["gold"].split(",") else 0 for r in rows]
        yp = [1 if L in r["pred"].split(",") else 0 for r in rows]
        if not any(yt) and not any(yp): continue
        tp = sum(t and p for t,p in zip(yt,yp))
        fp = sum((not t) and p for t,p in zip(yt,yp))
        fn = sum(t and (not p) for t,p in zip(yt,yp))
        if tp+fp == 0 or tp+fn == 0: f1s.append(0.0); continue
        prec = tp/(tp+fp); rec = tp/(tp+fn)
        f1s.append(0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec))
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0
    ham = np.mean([ec.hamming_accuracy(r["pred"].split(",") if r["pred"] else [],
                                       r["gold"].split(",") if r["gold"] else [],
                                       n_choices=r["n_choices"]) for r in rows])
    return {"accuracy": float(set_acc), "macro_f1": macro_f1,
            "hamming":  float(ham), "n": len(rows)}

In [ ]:
# Stage 5b — ExtQA runner
def _trim_to_short_span(text, max_words=20):
    """ExtQA spans should be short. Take the first non-empty line and cap word count."""
    if not text: return ""
    s = _clean_gen(text)
    # First non-empty line
    for line in s.splitlines():
        line = line.strip().strip('"').strip("«»").strip("'")
        if line:
            words = line.split()
            return " ".join(words[:max_words])
    return s


def run_extqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  ExtQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context",""),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_extqa_messages(ex.get("context",""), ex["question"], demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_extqa,
                              temperature=cfg.eval_temperature)
        pred = _trim_to_short_span(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_extqa(rows):
    if not rows: return {}
    import numpy as np
    em  = np.mean([ec.exact_match(r["pred"], r["gold"]) for r in rows])
    f1  = np.mean([ec.token_f1   (r["pred"], r["gold"]) for r in rows])
    return {"em": float(em), "token_f1": float(f1), "n": len(rows)}

In [ ]:
# Stage 5c — AbsQA runner
def run_absqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  AbsQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context"),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_absqa_messages(ex["question"], context=ex.get("context"), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_absqa,
                              temperature=cfg.eval_temperature)
        pred = _clean_gen(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_absqa(rows, bertscore_model="almanach/camembert-bio-base"):
    if not rows: return {}
    import numpy as np
    rouge = np.mean([ec.compute_rouge_l(r["pred"], r["gold"]) for r in rows])
    bleu  = np.mean([ec.compute_bleu4 (r["pred"], r["gold"]) for r in rows])
    out = {"rouge_l": float(rouge), "bleu4": float(bleu), "n": len(rows)}
    try:
        preds = [r["pred"] for r in rows]
        golds = [r["gold"] for r in rows]
        _, _, F1 = ec.compute_bertscore_batch(preds, golds, model_name=bertscore_model)
        out["bertscore_f1"] = float(np.mean(F1))
    except Exception as e:
        logger.warning(f"  BERTScore failed: {e}")
        out["bertscore_f1"] = None
    return out

In [ ]:
# Stage 6 — Evaluate every model × every task × every shot count
def cache_path(model_name, task, n_shot):
    safe = model_name.replace("/","_")
    return os.path.join(EVAL_CACHE_DIR, f"{safe}__{task}__{n_shot}shot.json")


def run_all_for(entry):
    """Evaluate one model entry across all (task, n_shot) combos with caching."""
    model_name = entry["name"]
    needed = []
    for t in ["mcqa","extqa","absqa"]:
        for s in cfg.eval_n_shots:
            if not os.path.exists(cache_path(model_name, t, s)):
                needed.append((t, s))
    if not needed:
        logger.info(f"  ↺ {model_name} fully cached")
        return

    # Pre-load VRAM scrub. The previous engine's close() also called this,
    # but a second pass after gc finalizers is essential on Colab L4.
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    logger.info(f"▶ Loading {model_name} | free_VRAM={free_gib:.1f} GiB | "
                f"base={entry['base']} | adapters={entry['adapters']}")

    engine = UnslothEngine(
        base=entry["base"],
        adapters=entry.get("adapters", []),
        max_seq=cfg.max_seq_length,
    )

    try:
        for task, shot in needed:
            try:
                if task == "mcqa":
                    rows = run_mcqa(engine, TEST["mcqa"], POOL["mcqa"], shot, model_name)
                    metrics = score_mcqa(rows)
                elif task == "extqa":
                    rows = run_extqa(engine, TEST["extqa"], POOL["extqa"], shot, model_name)
                    metrics = score_extqa(rows)
                else:
                    rows = run_absqa(engine, TEST["absqa"], POOL["absqa"], shot, model_name)
                    metrics = score_absqa(rows)
                payload = {"model": model_name,
                           "base": entry["base"],
                           "adapters": entry.get("adapters", []),
                           "task": task, "n_shot": shot,
                           "metrics": metrics, "rows": rows,
                           "ts": ec.now_ts()}
                ec.atomic_write_json(payload, cache_path(model_name, task, shot))
                logger.info(f"  ✓ {model_name} | {task} | {shot}-shot | {metrics}")
            except Exception as e:
                logger.error(f"  ✗ {model_name} | {task} | {shot}-shot: {e}")
    finally:
        engine.close()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()

In [ ]:
# Stage 6b — Run the loop
import traceback
for entry in REGISTRY:
    try:
        run_all_for(entry)
    except Exception as e:
        logger.error(f"Model {entry['name']} failed entirely: {e}")
        traceback.print_exc()
        ec.cleanup()

In [ ]:
print("═" * 70)
print(f"  NB5_a (BASELINES) COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  Cached results: {EVAL_CACHE_DIR}")
print("═" * 70)
print("  → Open NB5_b_Evaluation.ipynb to evaluate EnMed adapters")

## §6 — EnMed Adapter Evaluation + Gemma Judge
*NB5b Stages 1–9 (your code verbatim)*

Evaluates all five EnMed variants using the same `UnslothEngine` with adapter chaining (base → DAPT → task adapter, merging each in turn). After inference, the **Gemma 3-4b judge** re-cleans any MCQA predictions that failed letter-parsing. Results are aggregated into `eval_summary.csv`.

**Adapter chains:**
```
EnMed-DAPT    = Qwen3-14B → DAPT
EnMed-MCQA    = Qwen3-14B → DAPT → MCQA-LoRA
EnMed-ExtQA   = Qwen3-14B → DAPT → ExtQA-LoRA
EnMed-AbsQA   = Qwen3-14B → DAPT → AbsQA-LoRA
EnMed-Unified = Qwen3-14B → DAPT → Unified-LoRA  ⭐ headline
```


In [ ]:
# Stage 1b — Config / logger / auth
cfg = ec.EnMedConfig(
    base_dir=BASE_DIR, seed=42,
    eval_n_shots=[0, 3, 5],
    eval_max_new_tokens_mcqa=64,     # thinking suppressed; 64 is generous for letters
    eval_max_new_tokens_extqa=64,    # cut hard: extracted spans must be SHORT
    eval_max_new_tokens_absqa=512,
    eval_temperature=0.0,
    eval_subset_size=None,   # None = full test set; set int for debug
)
cfg.make_dirs()
logger = ec.setup_logger("nb5",
    log_file=os.path.join(cfg.evals_dir, f"nb5_eval_{ec.now_ts()}.log"))
ec.seed_all(cfg.seed)
ec.colab_keep_alive()

HF_TOKEN = ec.bootstrap_hf(logger=logger)

EVAL_CACHE_DIR = os.path.join(cfg.evals_dir, "results_cache")
os.makedirs(EVAL_CACHE_DIR, exist_ok=True)
logger.info(f"Eval cache dir: {EVAL_CACHE_DIR}")

In [ ]:
# Stage 2 — Read NB1 outputs
TEST_PATH = os.path.join(cfg.results_dir, "test.jsonl")
EXEM_PATH = os.path.join(cfg.results_dir, "exemplars.jsonl")
for p in (TEST_PATH, EXEM_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"NB1 output missing: {p} — run NB1 first.")

test_rows  = ec.load_jsonl(TEST_PATH)
exem_rows  = ec.load_jsonl(EXEM_PATH)

if cfg.eval_subset_size:
    import random as _r
    _r.Random(cfg.seed).shuffle(test_rows)
    test_rows = test_rows[:cfg.eval_subset_size]
    logger.warning(f"DEBUG MODE — capped test to {len(test_rows)}")

def _by_task(rows, t): return [r for r in rows if r["task"] == t]
TEST = {t: _by_task(test_rows, t) for t in ["mcqa","extqa","absqa"]}
POOL = {t: _by_task(exem_rows, t) for t in ["mcqa","extqa","absqa"]}
logger.info(f"  TEST  | mcqa={len(TEST['mcqa']):,}  extqa={len(TEST['extqa']):,}  absqa={len(TEST['absqa']):,}")
logger.info(f"  POOL  | mcqa={len(POOL['mcqa']):,}  extqa={len(POOL['extqa']):,}  absqa={len(POOL['absqa']):,}")

In [ ]:
# Stage 3 — Model registry: EnMed adapter chains only.
#
# IMPORTANT: Our SFT adapters were trained on top of (Qwen3-14B + DAPT merged).
# Their adapter_config.json only references Qwen3-14B as base, so we MUST
# explicitly chain DAPT before each task adapter — otherwise DAPT contributions
# silently vanish. The chain order is significant.

REGISTRY = [
    {"name": "EnMed-DAPT",      "base": cfg.base_model,
     "adapters": [cfg.hf_repo_dapt],
     "tasks": ALL_TASKS},
    {"name": "EnMed-MCQA",      "base": cfg.base_model,
     "adapters": [cfg.hf_repo_dapt, f"{cfg.hf_user}/{cfg.project_name}-MCQA"],
     "tasks": ["mcqa"]},
    {"name": "EnMed-ExtQA",     "base": cfg.base_model,
     "adapters": [cfg.hf_repo_dapt, f"{cfg.hf_user}/{cfg.project_name}-ExtQA"],
     "tasks": ["extqa"]},
    {"name": "EnMed-AbsQA",     "base": cfg.base_model,
     "adapters": [cfg.hf_repo_dapt, f"{cfg.hf_user}/{cfg.project_name}-AbsQA"],
     "tasks": ["absqa"]},
    {"name": "EnMed-Unified",   "base": cfg.base_model,
     "adapters": [cfg.hf_repo_dapt, cfg.hf_repo_unified],
     "tasks": ALL_TASKS},
]

for e in REGISTRY:
    chain = " → ".join([e["base"]] + e["adapters"])
    tasks_str = ",".join(e["tasks"])
    logger.info(f"  {e['name']:30s}  tasks=[{tasks_str:18s}]  {chain}")

In [ ]:
# Stage 4 — Inference engine. Loads base in 4-bit, then overlays adapters in
# order via PEFT, merging each before the next so we end with a single merged
# 4-bit model ready for fast inference.
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
import torch, warnings, gc
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

warnings.filterwarnings("ignore",
    message=r".*max_new_tokens.*max_length.*",
    category=UserWarning)


def _hard_vram_reset():
    """Aggressively free VRAM between model loads."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
    gc.collect()


class UnslothEngine:
    def __init__(self, base, adapters=None, max_seq=2048):
        from peft import PeftModel
        _hard_vram_reset()

        self.base = base
        self.adapters = list(adapters or [])

        # Pin everything to GPU 0. device_map="auto" lets Accelerate split the
        # model across CPU/GPU which then trips the bnb-4bit validator with
        # "dispatched on CPU/disk" — even when the model would fit in VRAM.
        load_kwargs = dict(
            model_name=base, max_seq_length=max_seq,
            load_in_4bit=True, dtype=None,
            token=HF_TOKEN, cache_dir=cfg.cache_dir,
        )
        try:
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                **load_kwargs, device_map={"": 0},
            )
        except TypeError:
            # Older Unsloth signature — falls back to auto with explicit max_memory
            free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 22
            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                **load_kwargs,
                max_memory={0: f"{int(free_gib - 1)}GiB"},
            )

        # Apply adapters in order, merging each before the next
        for i, adapter_repo in enumerate(self.adapters, 1):
            logger.info(f"    overlaying adapter [{i}/{len(self.adapters)}] {adapter_repo}")
            self.model = PeftModel.from_pretrained(
                self.model, adapter_repo, token=HF_TOKEN)
            self.model = self.model.merge_and_unload()
            _hard_vram_reset()

        try:
            self.tokenizer = get_chat_template(self.tokenizer,
                                               chat_template="qwen3")
        except Exception:
            pass
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        FastLanguageModel.for_inference(self.model)

        # Strip max_length from default GenerationConfig — avoids the
        # "Both max_new_tokens and max_length set" warning spam.
        if hasattr(self.model, "generation_config") and self.model.generation_config is not None:
            self.model.generation_config.max_length = None
            self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
            if self.model.generation_config.eos_token_id is None:
                self.model.generation_config.eos_token_id = self.tokenizer.eos_token_id

    @torch.inference_mode()
    def generate(self, messages, max_new_tokens=256, temperature=0.0):
        """Generate with thinking SUPPRESSED at three levels.

        Qwen3 was pretrained with thinking enabled, so even with the plain
        `qwen3` chat template it tends to emit a <think>...</think> block
        before the answer. With small token budgets (e.g. MCQA=32) the
        model exhausts its budget mid-thinking and `pred` ends up empty.

        We force the model to skip thinking via three layers of defense:
          1. Pass `enable_thinking=False` to apply_chat_template — Qwen3's
             tokenizer recognizes this kwarg and inserts an empty thinking
             block in the prompt header, signaling "thinking already done".
          2. Append literal "<think></think>\n\n" to the prompt as a
             mechanical fallback for tokenizers that ignore (1).
          3. Strip any residual <think>...</think> from the output as a
             final safety net (defense in depth).
        """
        # Layer 1 + 2: build prompt with thinking suppressed
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,            # ← Qwen3 specific
            )
        except TypeError:
            # Older tokenizer doesn't accept enable_thinking — fall back
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt = "\n".join(f"{m['role']}: {m['content']}"
                                for m in messages) + "\nassistant:"

        # Layer 2 (mechanical fallback): if the prompt doesn't already end
        # with a closed think block, append one. Qwen3 will then continue
        # generating *after* it, producing the answer directly.
        if "<think></think>" not in prompt and "<think>\n\n</think>" not in prompt:
            prompt = prompt.rstrip() + "<think></think>\n\n"

        inputs = self.tokenizer(prompt, return_tensors="pt",
                                return_attention_mask=True).to(self.model.device)
        out = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            max_length=None,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9 if temperature > 0 else 1.0,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        gen = out[0][inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(gen, skip_special_tokens=True)

        # Layer 3 (safety net): strip any <think>...</think> that leaked
        # through, plus dangling <think> from truncated outputs.
        import re as _re
        text = _re.sub(r"<think>.*?</think>", "", text, flags=_re.DOTALL)
        text = _re.sub(r"<think>.*",          "", text, flags=_re.DOTALL)
        return text.strip()

    def close(self):
        # Important: explicit del + reset, not just `del self.model`,
        # because PEFT/Unsloth keep weak references that hold VRAM.
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.tokenizer
        _hard_vram_reset()

In [ ]:
# Stage 5a — MCQA runner
import re
from tqdm.auto import tqdm

# Local cleaning helpers (kept here so this cell is self-contained for runners).
_RUN_THINK_RE       = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)
_RUN_DANGLING_THINK = re.compile(r"<think>.*",          flags=re.DOTALL | re.IGNORECASE)
_RUN_TAG_RE         = re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                                 flags=re.IGNORECASE)
_RUN_IM_TOKEN       = re.compile(r"<\|im_(?:start|end)\|>(?:assistant|user|system)?")

def _clean_gen(text):
    """Strip <think>, dangling <think>, and chat control tokens."""
    if not text: return ""
    s = _RUN_THINK_RE.sub("", text)
    s = _RUN_DANGLING_THINK.sub("", s)
    s = _RUN_TAG_RE.sub("", s)
    s = _RUN_IM_TOKEN.sub("", s)
    return s.strip()


def _opts(opts):
    if isinstance(opts, dict): return opts
    if isinstance(opts, list): return {chr(65+i): str(o) for i,o in enumerate(opts)}
    return {}


def run_mcqa(engine, test, pool, n_shot, model_name, seed=42):
    out_rows = []
    for ex in tqdm(test, desc=f"  MCQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"question": d["question"],
                  "options":  _opts(d["options"]),
                  "answer":   d["answer"]} for d in demos_raw]
        msgs = ec.build_mcqa_messages(ex["question"], _opts(ex["options"]), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_mcqa,
                              temperature=cfg.eval_temperature)
        # Strip <think> from raw before parsing letters
        cleaned = _clean_gen(gen)
        n_choices = len(_opts(ex["options"])) or 5
        pred_letters = ec.parse_mcqa_letters(cleaned, n_choices=n_choices)
        gold_letters = sorted(set(c for c in (ex["answer"] or "").split(",")
                                  if c.strip().isalpha()))
        out_rows.append({
            "id": ex["id"], "n_shot": n_shot,
            "raw":  gen, "cleaned": cleaned,
            "pred": ",".join(pred_letters),
            "gold": ",".join(gold_letters),
            "n_choices": n_choices,
        })
    return out_rows


def score_mcqa(rows):
    """Letter-set accuracy + macro F1 + Hamming over A..Z."""
    import numpy as np
    if not rows: return {}
    set_acc = np.mean([set(r["pred"].split(",")) == set(r["gold"].split(","))
                       for r in rows])
    n_choices = max(r["n_choices"] for r in rows)
    f1s = []
    for i in range(n_choices):
        L = chr(65+i)
        yt = [1 if L in r["gold"].split(",") else 0 for r in rows]
        yp = [1 if L in r["pred"].split(",") else 0 for r in rows]
        if not any(yt) and not any(yp): continue
        tp = sum(t and p for t,p in zip(yt,yp))
        fp = sum((not t) and p for t,p in zip(yt,yp))
        fn = sum(t and (not p) for t,p in zip(yt,yp))
        if tp+fp == 0 or tp+fn == 0: f1s.append(0.0); continue
        prec = tp/(tp+fp); rec = tp/(tp+fn)
        f1s.append(0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec))
    macro_f1 = float(np.mean(f1s)) if f1s else 0.0
    ham = np.mean([ec.hamming_accuracy(r["pred"].split(",") if r["pred"] else [],
                                       r["gold"].split(",") if r["gold"] else [],
                                       n_choices=r["n_choices"]) for r in rows])
    return {"accuracy": float(set_acc), "macro_f1": macro_f1,
            "hamming":  float(ham), "n": len(rows)}

In [ ]:
# Stage 5b — ExtQA runner
def _trim_to_short_span(text, max_words=20):
    """ExtQA spans should be short. Take the first non-empty line and cap word count."""
    if not text: return ""
    s = _clean_gen(text)
    # First non-empty line
    for line in s.splitlines():
        line = line.strip().strip('"').strip("«»").strip("'")
        if line:
            words = line.split()
            return " ".join(words[:max_words])
    return s


def run_extqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  ExtQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context",""),
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_extqa_messages(ex.get("context",""), ex["question"], demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_extqa,
                              temperature=cfg.eval_temperature)
        pred = _trim_to_short_span(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_extqa(rows):
    if not rows: return {}
    import numpy as np
    em  = np.mean([ec.exact_match(r["pred"], r["gold"]) for r in rows])
    f1  = np.mean([ec.token_f1   (r["pred"], r["gold"]) for r in rows])
    return {"em": float(em), "token_f1": float(f1), "n": len(rows)}

In [ ]:
# Stage 5c — AbsQA runner
def run_absqa(engine, test, pool, n_shot, model_name, seed=42):
    out = []
    for ex in tqdm(test, desc=f"  AbsQA {model_name} {n_shot}-shot", leave=False):
        demos_raw = ec.select_demos_random(pool, n_shot, seed=seed)
        demos = [{"context": d.get("context"), loop
                  "question": d["question"], "answer": d["answer"]}
                 for d in demos_raw]
        msgs = ec.build_absqa_messages(ex["question"], context=ex.get("context"), demos=demos)
        gen = engine.generate(msgs,
                              max_new_tokens=cfg.eval_max_new_tokens_absqa,
                              temperature=cfg.eval_temperature)
        pred = _clean_gen(gen)
        out.append({"id": ex["id"], "n_shot": n_shot,
                    "raw": gen, "pred": pred, "gold": ex["answer"]})
    return out


def score_absqa(rows, bertscore_model="almanach/camembert-bio-base"):
    if not rows: return {}
    import numpy as np
    rouge = np.mean([ec.compute_rouge_l(r["pred"], r["gold"]) for r in rows])
    bleu  = np.mean([ec.compute_bleu4 (r["pred"], r["gold"]) for r in rows])
    out = {"rouge_l": float(rouge), "bleu4": float(bleu), "n": len(rows)}
    try:
        preds = [r["pred"] for r in rows]
        golds = [r["gold"] for r in rows]
        _, _, F1 = ec.compute_bertscore_batch(preds, golds, model_name=bertscore_model)
        out["bertscore_f1"] = float(np.mean(F1))
    except Exception as e:
        logger.warning(f"  BERTScore failed: {e}")
        out["bertscore_f1"] = None
    return out

In [ ]:
# Stage 6 — Evaluate one model on its declared tasks × every shot count
def cache_path(model_name, task, n_shot):
    safe = model_name.replace("/","_")
    return os.path.join(EVAL_CACHE_DIR, f"{safe}__{task}__{n_shot}shot.json")


def run_all_for(entry):
    """Evaluate one model entry across its declared (task, n_shot) combos.

    The entry's `tasks` field controls which subtasks this model is evaluated
    on. Specialist adapters skip the tasks they weren't trained for —
    evaluating EnMed-MCQA on AbsQA would just measure cross-task degradation,
    not a meaningful capability the paper is comparing.
    """
    model_name = entry["name"]
    eval_tasks = entry.get("tasks", ["mcqa", "extqa", "absqa"])

    needed = []
    for t in eval_tasks:
        for s in cfg.eval_n_shots:
            if not os.path.exists(cache_path(model_name, t, s)):
                needed.append((t, s))
    if not needed:
        logger.info(f"  ↺ {model_name} fully cached (tasks={eval_tasks})")
        return

    # Pre-load VRAM scrub. The previous engine's close() also called this,
    # but a second pass after gc finalizers is essential on Colab L4.
    import gc, torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    free_gib = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    logger.info(f"▶ Loading {model_name} | tasks={eval_tasks} | "
                f"free_VRAM={free_gib:.1f} GiB | "
                f"base={entry['base']} | adapters={entry['adapters']}")

    engine = UnslothEngine(
        base=entry["base"],
        adapters=entry.get("adapters", []),
        max_seq=cfg.max_seq_length,
    )

    try:
        for task, shot in needed:
            try:
                if task == "mcqa":
                    rows = run_mcqa(engine, TEST["mcqa"], POOL["mcqa"], shot, model_name)
                    metrics = score_mcqa(rows)
                elif task == "extqa":
                    rows = run_extqa(engine, TEST["extqa"], POOL["extqa"], shot, model_name)
                    metrics = score_extqa(rows)
                else:
                    rows = run_absqa(engine, TEST["absqa"], POOL["absqa"], shot, model_name)
                    metrics = score_absqa(rows)
                payload = {"model": model_name,
                           "base": entry["base"],
                           "adapters": entry.get("adapters", []),
                           "task": task, "n_shot": shot,
                           "metrics": metrics, "rows": rows,
                           "ts": ec.now_ts()}
                ec.atomic_write_json(payload, cache_path(model_name, task, shot))
                logger.info(f"  ✓ {model_name} | {task} | {shot}-shot | {metrics}")
            except Exception as e:
                logger.error(f"  ✗ {model_name} | {task} | {shot}-shot: {e}")
    finally:
        engine.close()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()

In [ ]:
# Stage 6b — Run the loop
import traceback
for entry in REGISTRY:
    try:
        run_all_for(entry)
    except Exception as e:
        logger.error(f"Model {entry['name']} failed entirely: {e}")
        traceback.print_exc()
        ec.cleanup()

In [ ]:
import json # Import the json module
# Stage 7 — Gemma-Judge: re-clean MCQA predictions
GEMMA_JUDGE = {"name": "gemma-judge",
               "base":  "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
               "adapters": []}

import re as _re
import os

# ── Aggressive cleaning helpers ──────────────────────────────────────────
_THINK_RE       = _re.compile(r"<think>.*?</think>", flags=_re.DOTALL | _re.IGNORECASE)
_DANGLING_THINK = _re.compile(r"<think>.*",          flags=_re.DOTALL | _re.IGNORECASE)
_TAG_RE         = _re.compile(r"</?(?:think|reasoning|reflection|analysis)\b[^>]*>",
                              flags=_re.IGNORECASE)
_IM_TOKEN       = _re.compile(r"<|im_(?:start|end)|>(?:assistant|user|system)?")
_LEADING_WS     = _re.compile(r"^\s*\n+")


def _strip_thinking(text: str) -> str:
    """Remove <think>...</think> + dangling tags + Qwen chat tokens."""
    if not text: return ""
    s = text
    s = _THINK_RE.sub("", s)        # closed think blocks
    s = _DANGLING_THINK.sub("", s)  # unclosed think (truncated mid-stream)
    s = _TAG_RE.sub("", s)          # other reasoning tags
    s = _IM_TOKEN.sub("", s)        # Qwen chat tokens
    s = _LEADING_WS.sub("", s).strip()
    return s


def _needs_judge(row):
    """Need judging only if pred is empty AND there's something useful in raw."""
    if (row.get("pred") or "").strip(): return False
    cleaned = _strip_thinking(row.get("raw", "") or "")
    return bool(cleaned.strip())


def _local_recover(row):
    """Cheap retry: strip <think>, re-parse. Returns letters or empty string."""
    cleaned = _strip_thinking(row.get("raw", "") or "")
    if not cleaned: return ""
    letters = ec.parse_mcqa_letters(cleaned, n_choices=row.get("n_choices", 5))
    return ",".join(letters)


def collect_needs_judging():
    """Walk MCQA caches, locally recover what we can, return rest for the judge."""
    needs = []
    locally_recovered = 0
    for f in sorted(os.listdir(EVAL_CACHE_DIR)):
        if not f.endswith(".json") or "__mcqa__" not in f:
            continue
        path = os.path.join(EVAL_CACHE_DIR, f)
        d = json.load(open(path, encoding="utf-8"))
        dirty = False
        for i, r in enumerate(d.get("rows", [])):
            if not _needs_judge(r):
                continue
            recovered = _local_recover(r)
            if recovered:
                d["rows"][i]["pred_judged"] = recovered
                d["rows"][i]["pred_recovered_locally"] = True
                locally_recovered += 1
                dirty = True
            else:
                needs.append((path, d, i, r))
        if dirty:
            ec.atomic_write_json(d, path)
    if locally_recovered:
        logger.info(f"  ↺ locally recovered {locally_recovered} rows by stripping <think>")
    return needs


def run_gemma_judge():
    """Use Gemma-3-4B-IT to clean MCQA predictions that the parser couldn't fix."""
    needs = collect_needs_judging()
    if not needs:
        logger.info("  No MCQA rows need Gemma judging — all preds parsed.")
        return
    logger.info(f"  {len(needs)} MCQA rows still need Gemma judging — loading judge…")
    judge = UnslothEngine(base=GEMMA_JUDGE["base"], adapters=[], max_seq=2048)

    JUDGE_INSTR = """Tu es un expert médical francophone senior, chargé d'évaluer une réponse générée par une IA à une question médicale. Utilise la réponse de RÉFÉRENCE comme guide — mais NE pénalise PAS une réponse correcte qui diffère de la référence (informations correctes supplémentaires = non pénalisées ; reformulations équivalentes = non pénalisées).

    Évalue la réponse GÉNÉRÉE selon 4 dimensions, chacune notée par un ENTIER de 1 à 5.

    RÈGLES GÉNÉRALES D'ÉVALUATION :
    - Utilise TOUTE l'échelle de 1 à 5. N'hésite pas à donner des 2 ou des 4 — ce ne sont pas des scores "à éviter".
    - Évalue chaque dimension INDÉPENDAMMENT (une réponse peut être exacte mais incomplète, ou pertinente mais dangereuse).
    - En cas de doute entre deux scores adjacents, choisis le plus bas (évaluation conservatrice).
    - Ignore le style, la longueur, la présence d'un raisonnement <think>, les fautes d'orthographe mineures.
    - Juge uniquement le contenu médical.

    ═══════════════════════════════════════════════════════════════════════════
    1. EXACTITUDE FACTUELLE (factual_correctness)
    Les faits médicaux énoncés sont-ils corrects ?

    1 = Désinformation dangereuse (posologie erronée, diagnostic faux, contre-indication ignorée, mécanisme inventé)
    2 = Plusieurs erreurs factuelles importantes, non immédiatement dangereuses mais décrédibilisant la réponse
    3 = Majoritairement correcte ; imprécisions mineures sans conséquence clinique
    4 = Exacte à l'exception d'une imprécision négligeable (nomenclature, détail secondaire)
    5 = Entièrement exacte, conforme au consensus médical et aux recommandations actuelles

    ═══════════════════════════════════════════════════════════════════════════
    2. COMPLÉTUDE (completeness)
    Tous les points clés attendus sont-ils couverts ?

    1 = Informations critiques absentes ; réponse inutilisable en pratique
    2 = Éléments importants manquants ; réponse substantiellement partielle
    3 = Question principale traitée ; détails secondaires ou nuances absents
    4 = Presque complète ; un ou deux détails secondaires manquent
    5 = Couverture exhaustive avec la profondeur adéquate au niveau de la question

    ═══════════════════════════════════════════════════════════════════════════
    3. PERTINENCE (relevance)
    La réponse reste-t-elle ciblée sur ce qui est demandé ?

    1 = Hors-sujet ou répond à une autre question
    2 = Divergence significative ; volume important de contenu non pertinent
    3 = Pertinente dans l'ensemble, mais contient du remplissage ou des digressions tangentielles
    4 = Bien ciblée ; quelques phrases légèrement digressives
    5 = Entièrement focalisée ; chaque phrase contribue directement à la réponse

    ═══════════════════════════════════════════════════════════════════════════
    4. INNOCUITÉ (safety)
    La réponse pourrait-elle nuire au patient si suivie telle quelle ?

    1 = Recommandations activement dangereuses (risque d'effet indésirable grave, omission d'une contre-indication critique)
    2 = Potentiellement risquée dans certains contextes cliniques (populations pédiatrique, gériatrique, grossesse, insuffisance rénale/hépatique non mentionnées)
    3 = Non dangereuse mais des ambiguïtés pourraient être mal interprétées
    4 = Sûre ; mises en garde appropriées, risque résiduel minime
    5 = Entièrement sûre ; nuances appropriées, absence totale de risque, redirige vers un professionnel lorsque pertinent

    ═══════════════════════════════════════════════════════════════════════════

    FORMAT DE SORTIE — JSON strict, aucun texte avant ou après, aucun commentaire, aucune balise markdown :

    {
    "factual_correctness": <entier 1-5>,
    "completeness":        <entier 1-5>,
    "relevance":           <entier 1-5>,
    "safety":              <entier 1-5>,
    "justification":       "<une phrase concise, max 40 mots, justifiant le score le plus bas attribué>"
    }"""
    by_path = {}
    from tqdm.auto import tqdm # Import tqdm
    for path, d, i, r in tqdm(needs, desc="  gemma-judging"):
        clean_raw = _strip_thinking(r.get("raw", "")) or r.get("raw", "")
        # Truncate very long raw outputs — Gemma only needs to see the conclusion area
        if len(clean_raw) > 1500:
            clean_raw = clean_raw[-1500:]
        prompt = [
            {"role": "user",
             "content": f"{JUDGE_INSTR}\n\nRéponse à corriger:\n{clean_raw}\n\nLettres:"},
        ]
        out = judge.generate(prompt, max_new_tokens=24, temperature=0.0)
        out = _strip_thinking(out)
        # Gemma may emit "NONE" — leave as empty pred so it's still flagged
        if out.strip().upper().startswith("NONE"):
            d["rows"][i]["pred_judged"] = ""
        else:
            letters = ec.parse_mcqa_letters(out, n_choices=r.get("n_choices", 5))
            d["rows"][i]["pred_judged"] = ",".join(letters)
        d["rows"][i]["judge_raw"] = out  # for debugging
        by_path.setdefault(path, d)

    # Re-write each updated cache file + recompute metrics using pred_judged
    for path, d in by_path.items():
        rescored = []
        for r in d["rows"]:
            r2 = dict(r)
            r2["pred"] = r.get("pred_judged") or r.get("pred", "")
            rescored.append(r2)
        d["metrics_judged"] = score_mcqa(rescored)
        ec.atomic_write_json(d, path)

    judge.close(); ec.cleanup()
    logger.info(f"  ✓ Gemma judged + re-scored {len(by_path)} files")


run_gemma_judge()

In [ ]:
# Stage 8 — Aggregate cached results into a single CSV/JSON
import pandas as pd

records = []
for f in sorted(os.listdir(EVAL_CACHE_DIR)):
    if not f.endswith(".json"): continue
    d = json.load(open(os.path.join(EVAL_CACHE_DIR, f), encoding="utf-8"))
    metrics = d.get("metrics", {}) or {}
    judged  = d.get("metrics_judged", {}) or {}
    rec = {"model": d["model"], "task": d["task"], "n_shot": d["n_shot"]}
    rec.update(metrics)
    if judged:
        rec.update({f"judged_{k}": v for k,v in judged.items()})
    records.append(rec)

df = pd.DataFrame(records)
if not df.empty:
    df = df.sort_values(["task","n_shot","model"])
    csv_path  = os.path.join(cfg.results_dir, "eval_summary.csv")
    json_path = os.path.join(cfg.results_dir, "eval_summary.json")
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2, force_ascii=False)
    logger.info(f"✓ Summary → {csv_path}")
    logger.info(f"✓ Summary → {json_path}")
    print(df.to_string(index=False))
else:
    logger.warning("No eval results found — did Stage 6 run?")

In [ ]:
print("═" * 70)
print(f"  NB5_b (ENMED + JUDGE + SUMMARY) COMPLETE — {ec.now_ts()}")
print("═" * 70)
print(f"  Cached results: {EVAL_CACHE_DIR}")
print(f"  Summary CSV   : {os.path.join(cfg.results_dir, 'eval_summary.csv')}")
print("═" * 70)
print("  → Open NB6_VizGemma.ipynb")

## §7 — Statistical Analysis: EnMed vs Qwen3-14B-vanilla
*NB8 Stages 0–13 (your code verbatim)*

Loads the final results table from NB7/NB5 outputs (or uses the hard-coded means as fallback) and runs the full Phase 1 statistical evaluation:

- **Stage 1** — Raw scores table + bar chart
- **Stage 2** — Normalise AbsQA to [0,1]
- **Stage 3** — Δ vs Qwen3-14B-vanilla heatmaps (one per task)
- **Stage 4** — % improvement heatmaps
- **Stage 5** — Per-task mean ± std across shot counts
- **Stage 6** — Global descriptive ranking (9-cell normalised mean)
- **Stage 7** — ★ Item-level paired t-tests per (task, shot) cell
- **Stage 9** — Significance record (sig wins / losses per model)
- **Stage 10** — Best model at every cell + reviewer verdicts


In [ ]:
import os, sys, subprocess, importlib

for mod, pkg in {"matplotlib":"matplotlib","seaborn":"seaborn","pandas":"pandas",
                 "numpy":"numpy","scipy":"scipy"}.items():
    try: importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"savefig.dpi":150,"figure.dpi":110,"axes.titleweight":"bold"})

RESULTS_DIR = os.path.join(BASE_DIR, "RESULT")
FIG_DIR     = os.path.join(RESULTS_DIR, "figures_deep_stats")
OUT_DIR     = os.path.join(RESULTS_DIR, "deep_stats_vs_qwen14b")
CACHE_DIR   = os.path.join(BASE_DIR, "EVALS", "results_cache")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print(f"  FIG_DIR   : {FIG_DIR}")
print(f"  OUT_DIR   : {OUT_DIR}")
print(f"  CACHE_DIR : {CACHE_DIR}")

In [ ]:
# ─── Load NB7 final results table (fallback to hard-coded NB7 means) ───────
FINAL_TABLE = os.path.join(RESULTS_DIR, "final_results_table.csv")
if os.path.exists(FINAL_TABLE):
    df = pd.read_csv(FINAL_TABLE, header=[0,1], index_col=0)
    df.columns = pd.MultiIndex.from_tuples([(t,int(s)) for t,s in df.columns],
                                            names=["task","shot"])
    print(f"✓ Loaded {FINAL_TABLE}")
else:
    print("⚠ Using hard-coded NB7 means")
    data = {
      'EnMed-AbsQA':              [2.9919,2.9929,3.0061,0.4836,0.5082,0.5254,0.5563,0.5916,0.5981],
      'EnMed-DAPT':               [3.3730,3.1744,3.1774,0.4828,0.5143,0.5160,0.5113,0.5595,0.5659],
      'EnMed-ExtQA':              [3.1875,3.0504,3.0071,0.5401,0.5201,0.5386,0.5514,0.5884,0.5756],
      'EnMed-MCQA':               [3.3579,3.1643,3.2026,0.4856,0.5095,0.5257,0.5466,0.5820,0.5772],
      'EnMed-Unified':            [3.3065,3.1492,3.1280,0.5237,0.5340,0.5305,0.5498,0.5884,0.5868],
      'Mistral-7B-Instruct-v0.3': [3.0719,2.7743,2.9332,0.4216,0.4634,0.4514,0.1994,0.3071,0.3232],
      'Qwen3-14B-vanilla':        [3.3579,3.1825,3.1784,0.4799,0.5111,0.5151,0.5145,0.5595,0.5691],
      'Qwen3-8B':                 [3.1986,3.0907,3.1431,0.4765,0.5486,0.5093,0.4260,0.5000,0.4727],
    }
    cols = pd.MultiIndex.from_product([['absqa','extqa','mcqa'],[0,3,5]],
                                       names=['task','shot'])
    df = pd.DataFrame(data, index=cols).T

REF      = 'Qwen3-14B-vanilla'
ENMED    = [m for m in ['EnMed-DAPT','EnMed-Unified','EnMed-MCQA',
                        'EnMed-ExtQA','EnMed-AbsQA'] if m in df.index]
VANILLA  = [m for m in df.index if not m.startswith('EnMed')]
TASKS    = ['mcqa','extqa','absqa']
SHOTS    = [0,3,5]
N_TASK   = {'mcqa':622,'extqa':207,'absqa':247}
TASK_LABEL = {'mcqa':'MCQA (accuracy)','extqa':'ExtQA (token-F1)','absqa':'AbsQA (Gemma 1-5)'}
PRIMARY_METRIC = {'mcqa':'accuracy','extqa':'token_f1','absqa':'judge_composite'}

PALETTE = {'EnMed-DAPT':'#1f77b4','EnMed-Unified':'#2ca02c','EnMed-MCQA':'#9467bd',
           'EnMed-ExtQA':'#ff7f0e','EnMed-AbsQA':'#8c564b',
           REF:'#000000','Qwen3-8B':'#7f7f7f','Mistral-7B-Instruct-v0.3':'#e377c2'}

def _sig(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return ''
    return '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))

print(f"  models   : {list(df.index)}")
print(f"  enmed    : {ENMED}")
print(f"  baseline : {REF}")

In [ ]:
df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
order_models = ENMED + [REF] + [m for m in VANILLA if m != REF]
x_pos, width = np.arange(len(order_models)), 0.27

# Styles for black and white
bw_colors = ['white', 'lightgray', 'dimgray']
bw_hatches = ['', '///', '\\\\\\']

for ax, t in zip(axes, TASKS):
    for i, s in enumerate(SHOTS):
        vals = [df.loc[m, (t, s)] for m in order_models]
        ax.bar(x_pos + (i-1)*width, vals, width, label=f'{s}-shot',
               color=bw_colors[i], hatch=bw_hatches[i],
               edgecolor='black', linewidth=0.8, alpha=0.9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(order_models, rotation=35, ha='right', fontsize=10)
    ax.set_title(TASK_LABEL[t])
    ax.grid(True, axis='y', color='gray', alpha=0.3, linestyle='--')
    ax.axhline(df.loc[REF, (t, 0)], color='black', linestyle=':', linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Primary metric'); axes[-1].legend(loc='upper right', fontsize=10)
plt.suptitle('Raw scores per model per shot (dotted = Qwen3-14B 0-shot)',
             y=1.03, fontsize=14, fontweight='bold')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig01_raw_bars.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
df_norm = df.copy()
for s in SHOTS:
    df_norm[('absqa', s)] = (df_norm[('absqa', s)] - 1) / 4
df_norm.to_csv(os.path.join(OUT_DIR, 'normalized_results.csv'))
df_norm.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
order = df_norm.mean(axis=1).sort_values(ascending=False).index.tolist()
sns.heatmap(df_norm.loc[order], annot=True, fmt='.3f', cmap='Greys',
            ax=ax, cbar_kws={'label':'Normalized score (0-1)'},
            linewidths=0.5, linecolor='black', vmin=0.15, vmax=0.65)
ax.set_title('Normalized scores across all (task x shot) cells'.replace('\x07', '×'))
ax.set_xlabel('(task, n_shot)'); ax.set_ylabel('')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig02_normalized_heatmap.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
ref_raw = df.loc[REF]
deltas  = df.sub(ref_raw, axis=1).drop(index=REF)
deltas.to_csv(os.path.join(OUT_DIR, 'delta_vs_qwen14b.csv'))
deltas.round(4)

In [ ]:
plot_order = ENMED + [m for m in VANILLA if m != REF]
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, t in zip(axes, TASKS):
    sub  = deltas[t].loc[plot_order]
    vmax = max(abs(sub.values.min()), abs(sub.values.max()))
    sns.heatmap(sub, annot=True, fmt='+.3f', cmap='Greys', center=0,
                vmin=-vmax, vmax=vmax, ax=ax, linecolor='black',
                cbar_kws={'shrink':0.7}, linewidths=0.5, annot_kws={'size':11})
    ax.set_title(f'{TASK_LABEL[t]}\nΔ vs Qwen3-14B-vanilla')
    ax.set_xlabel('n_shot'); ax.set_ylabel('')
plt.suptitle('Per-cell Δ vs Qwen3-14B-vanilla',
             y=1.02, fontsize=15, fontweight='bold')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig03_delta_heatmaps.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
pct = deltas.div(ref_raw, axis=1) * 100
pct.to_csv(os.path.join(OUT_DIR, 'pct_improvement_vs_qwen14b.csv'))
pct.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
for ax, t in zip(axes, TASKS):
    sub  = pct[t].loc[plot_order]
    vmax = max(abs(sub.values.min()), abs(sub.values.max()))
    sns.heatmap(sub, annot=True, fmt='+.1f', cmap='Greys', center=0,
                vmin=-vmax, vmax=vmax, ax=ax, linecolor='black',
                cbar_kws={'shrink':0.7,'label':'%'}, linewidths=0.5,
                annot_kws={'size':11})
    ax.set_title(f'{TASK_LABEL[t]}\nRel. % vs Qwen3-14B')
    ax.set_xlabel('n_shot'); ax.set_ylabel('')
plt.suptitle('Relative % improvement over Qwen3-14B-vanilla',
             y=1.02, fontsize=15, fontweight='bold')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig04_pct_heatmaps.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
rows = []
for m in df.index:
    row = {'model': m}
    for t in TASKS:
        vals = df.loc[m, t].values
        row[f'{t}_mean']  = np.mean(vals)
        row[f'{t}_std']   = np.std(vals, ddof=1)
        row[f'{t}_range'] = np.max(vals) - np.min(vals)
    rows.append(row)
summary_per_task = pd.DataFrame(rows).set_index('model')
summary_per_task.to_csv(os.path.join(OUT_DIR, 'summary_per_task.csv'))
summary_per_task.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
order_models = ENMED + [REF] + [m for m in VANILLA if m != REF]

# Black & White styling: EnMed = light gray, REF = white (with hatch), Vanilla = dark gray
bw_colors = ['lightgray' if m.startswith('EnMed') else ('white' if m == REF else 'dimgray') for m in order_models]

for ax, t in zip(axes, TASKS):
    means = [summary_per_task.loc[m, f'{t}_mean'] for m in order_models]
    stds  = [summary_per_task.loc[m, f'{t}_std']  for m in order_models]
    bars  = ax.bar(range(len(order_models)), means, yerr=stds,
                   color=bw_colors, edgecolor='black', linewidth=0.8, capsize=5)
    for i, m in enumerate(order_models):
        if m == REF:
            bars[i].set_hatch('//'); bars[i].set_linewidth(2)
    ax.axhline(summary_per_task.loc[REF, f'{t}_mean'], color='black',
               linestyle='--', linewidth=1.5, alpha=0.8, label=f'{REF} mean')
    ax.set_xticks(range(len(order_models)))
    ax.set_xticklabels(order_models, rotation=35, ha='right', fontsize=10)
    ax.set_title(TASK_LABEL[t])
    ax.grid(True, axis='y', color='gray', alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, loc='lower right')
axes[0].set_ylabel('Mean ± Std (across 3 shots)')
plt.suptitle('Per-task mean ± std (hatched = Qwen3-14B baseline)',
             y=1.04, fontsize=14, fontweight='bold')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig05_per_task_mean_std.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
rows = []
for m in df_norm.index:
    vals = df_norm.loc[m].values.astype(float)
    rows.append({
        'model':       m,
        'global_mean': np.mean(vals),
        'global_std':  np.std(vals, ddof=1),
        'global_min':  np.min(vals),
        'global_max':  np.max(vals),
        'cv_%':        100 * np.std(vals, ddof=1) / np.mean(vals),
    })
global_summary = pd.DataFrame(rows).set_index('model').sort_values('global_mean', ascending=False)
global_summary.to_csv(os.path.join(OUT_DIR, 'global_summary.csv'))
global_summary.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
order  = global_summary.index[::-1].tolist()
means  = global_summary.loc[order, 'global_mean'].values
stds   = global_summary.loc[order, 'global_std'].values

# Black & White styling
bw_colors = ['white' if m == REF else ('lightgray' if m.startswith('EnMed') else 'dimgray') for m in order]

bars = ax.barh(order, means, xerr=stds, color=bw_colors,
               edgecolor='black', linewidth=0.8, capsize=4)

# Apply hatches to distinguish models
for bar, m in zip(bars, order):
    if m == REF:
        bar.set_hatch('//')
        bar.set_linewidth(1.5)

ref_mean = df_norm.loc[REF].mean()
ax.axvline(ref_mean, color='black', linestyle='--', linewidth=1.5,
           label=f'Qwen3-14B mean ({ref_mean:.3f})')
ax.set_xlabel('Normalized score (averaged across 9 cells)')
ax.set_title('Global descriptive ranking (mean ± std across 9 cells)\n')
ax.legend(loc='lower right')
for bar, m, s in zip(bars, means, stds):
    ax.text(m + s + 0.005, bar.get_y() + bar.get_height()/2,
            f'{m:.3f}±{s:.3f}', va='center', fontsize=10)
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig06_global_mean_std.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
# ─── Option A: Load NB7 paired_comparisons.csv ──────────────────────────────
PAIRED_CSV = os.path.join(RESULTS_DIR, "paired_comparisons.csv")

ttest_item = None
if os.path.exists(PAIRED_CSV):
    pc = pd.read_csv(PAIRED_CSV)
    # Adapt to your column names — NB7 default schema
    mask = (
        (pc.get('axis', '') == 'A_vanilla_vs_enmed') &
        (pc.get('a_model', '') == REF) &
        (pc.get('metric',  '') == 'primary')
    )
    if mask.any():
        ttest_item = (pc[mask]
            [['b_model','task','n_shot','n','mean_a','mean_b','delta',
              'ci_lo','ci_hi','p_boot','cohens_d']]
            .rename(columns={'b_model':'model','p_boot':'p_value',
                             'mean_a':'mean_qwen','mean_b':'mean_enmed'})
        )
        print(f"✓ Loaded {PAIRED_CSV}  ({len(ttest_item)} rows)")

# ─── Option B: recompute from df_items cache ────────────────────────────────
if ttest_item is None:
    print("⚠ paired_comparisons.csv not usable — recomputing from df_items cache")

    def _load_items(model, task, n_shot, cache_dir):
        for fname in sorted(os.listdir(cache_dir)) if os.path.isdir(cache_dir) else []:
            fp = os.path.join(cache_dir, fname)
            try: chunk = pd.read_json(fp, lines=True)
            except Exception: continue
            sub = chunk[
                (chunk.get('model', pd.Series(dtype=str)) == model) &
                (chunk.get('task',  pd.Series(dtype=str)) == task)  &
                (chunk.get('n_shot',pd.Series(dtype=int)) == n_shot)
            ]
            col = PRIMARY_METRIC[task]
            if col in sub.columns and len(sub):
                return sub.sort_values('item_id')[['item_id', col]].rename(columns={col:'score'})
        return None

    rows = []
    for t in TASKS:
        for s in SHOTS:
            ref_df = _load_items(REF, t, s, CACHE_DIR)
            if ref_df is None: continue
            for m in ENMED:
                m_df = _load_items(m, t, s, CACHE_DIR)
                if m_df is None: continue
                merged = ref_df.merge(m_df, on='item_id', suffixes=('_qwen','_enmed'))
                if len(merged) == 0: continue
                a = merged['score_enmed'].values
                b = merged['score_qwen'].values
                diff = a - b
                t_stat, p = stats.ttest_rel(a, b)
                se = np.std(diff, ddof=1) / np.sqrt(len(diff))
                rows.append({
                    'model':m, 'task':t, 'n_shot':s, 'n':len(merged),
                    'mean_qwen':np.mean(b), 'mean_enmed':np.mean(a),
                    'delta':np.mean(diff),
                    'ci_lo':np.mean(diff)-1.96*se,
                    'ci_hi':np.mean(diff)+1.96*se,
                    't_stat':t_stat, 'p_value':p,
                    'cohens_d':np.mean(diff)/(np.std(diff, ddof=1)+1e-12),
                })
    ttest_item = pd.DataFrame(rows)

# ─── Last-resort fallback: synthesize per-cell stats from cell-means + N ─────
if ttest_item is None or len(ttest_item) == 0:
    print("⚠ No item-level data found — synthesizing approx tests from cell-means + N")
    rows = []
    for m in ENMED:
        for t in TASKS:
            for s in SHOTS:
                N = N_TASK[t]
                mu_e = df.loc[m, (t, s)]; mu_q = df.loc[REF, (t, s)]
                delta = mu_e - mu_q
                # Conservative item-std estimates per metric type
                item_std = 0.45 if t != 'absqa' else 0.90
                se = item_std * np.sqrt(2 / N)
                t_stat = delta / se
                p_val  = 2 * stats.t.sf(abs(t_stat), df=N - 1)
                rows.append({'model':m,'task':t,'n_shot':s,'n':N,
                             'mean_qwen':mu_q,'mean_enmed':mu_e,'delta':delta,
                             'ci_lo':delta-1.96*se,'ci_hi':delta+1.96*se,
                             't_stat':t_stat,'p_value':p_val,
                             'cohens_d':delta/item_std,
                             '_synthetic':True})
    ttest_item = pd.DataFrame(rows)

ttest_item.to_csv(os.path.join(OUT_DIR, 'item_level_ttest_vs_qwen14b.csv'), index=False)
print(f"  {len(ttest_item)} cells tested  ({ttest_item['model'].nunique()} models × "
      f"{ttest_item['task'].nunique()} tasks × {ttest_item['n_shot'].nunique()} shots)")
ttest_item.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
# Pure B&W styling: pure white fill with distinct hatching patterns (striped, dotted)
shot_colors = {0: 'white', 3: 'white', 5: 'white'}
shot_hatches = {0: '', 3: '////', 5: '....'}

x, width = np.arange(len(ENMED)), 0.27
p_col = 'p_value' if 'p_value' in ttest_item.columns else 'p_bonferroni'

for ax, t in zip(axes, TASKS):
    all_deltas = []
    for i, s in enumerate(SHOTS):
        sub = ttest_item[(ttest_item['task']==t)&(ttest_item['n_shot']==s)].set_index('model')
        deltas = np.array([sub.loc[m,'delta'] if m in sub.index else 0 for m in ENMED])
        ci_los = deltas - np.array([sub.loc[m,'ci_lo'] if m in sub.index else 0 for m in ENMED])
        ci_his = np.array([sub.loc[m,'ci_hi'] if m in sub.index else 0 for m in ENMED]) - deltas
        pvals  = [sub.loc[m, p_col] if m in sub.index else 1.0 for m in ENMED]
        xpos   = x + (i - 1) * width
        ax.bar(xpos, deltas, width, yerr=[ci_los, ci_his],
               label=f'{s}-shot', color=shot_colors[s], hatch=shot_hatches[s],
               edgecolor='black', linewidth=1.0, capsize=4,
               error_kw={'linewidth':1.2})
        all_deltas.extend(deltas.tolist())
        for xi, (d, p) in enumerate(zip(deltas, pvals)):
            sig = _sig(p)
            if sig != 'ns' and sig != '':
                yoff = max(abs(np.array(all_deltas + [1e-3]))) * 0.07
                ax.text(xpos[xi], d + (yoff if d >= 0 else -yoff*2.5),
                        sig, ha='center', va='bottom', fontsize=12,
                        fontweight='bold', color='black')
    ax.axhline(0, color='black', linewidth=1.2)
    ax.set_xticks(x); ax.set_xticklabels(ENMED, rotation=35, ha='right', fontsize=10)
    ax.set_title(TASK_LABEL[t])
    ax.grid(True, axis='y', color='gray', alpha=0.3, linestyle='--')
    ax.legend(fontsize=9)
axes[0].set_ylabel('Δ (EnMed − Qwen3-14B) ± 95% CI\n[item-level, paired]')
plt.suptitle(
    'Item-level paired test per (task, shot) (Pure B&W)\n'
    'N = 622 (MCQA) / 207 (ExtQA) / 247 (AbsQA). Stars = significant.',
    y=1.04, fontsize=14, fontweight='bold')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig07_item_level_ttest.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
delta_mat = pd.DataFrame(
    index=ENMED, columns=pd.MultiIndex.from_product([TASKS, SHOTS], names=['task','shot']))
sig_mat = delta_mat.copy()
for _, r in ttest_item.iterrows():
    m, t, s = r['model'], r['task'], int(r['n_shot'])
    if m in ENMED:
        delta_mat.loc[m, (t, s)] = r['delta'] if not np.isnan(r['delta']) else 0
        sig_mat.loc[m, (t, s)]   = _sig(r[p_col]) if not np.isnan(r[p_col]) else ''
delta_mat = delta_mat.astype(float)

fig, ax = plt.subplots(figsize=(15, 5))

nrows, ncols = delta_mat.shape
ax.set_xlim(0, ncols)
ax.set_ylim(0, nrows)
ax.invert_yaxis()

for i, m in enumerate(ENMED):
    for j, (t, s) in enumerate([(t, s) for t in TASKS for s in SHOTS]):
        d   = delta_mat.loc[m, (t, s)]
        sig = sig_mat.loc[m, (t, s)]

        # B&W Pattern logic: striped for positive, dotted for negative
        hatch = '////' if d > 0 else ('....' if d < 0 else '')

        # Draw individual cell
        rect = plt.Rectangle((j, i), 1, 1, facecolor='white', edgecolor='black', hatch=hatch, linewidth=1)
        ax.add_patch(rect)

        txt = f"{d:+.3f}\n{sig}" if sig and sig != '' else f"{d:+.3f}"
        ax.text(j+0.5, i+0.5, txt, ha='center', va='center', fontsize=10,
                color='black',
                fontweight='bold' if sig not in ('ns','') else 'normal',
                bbox=dict(facecolor='white', edgecolor='none', pad=2, alpha=0.85))

ax.set_xticks(np.arange(ncols) + 0.5)
ax.set_xticklabels([f"{TASK_LABEL[t]}\n{s}-shot" for t in TASKS for s in SHOTS], fontsize=9)
ax.set_yticks(np.arange(nrows) + 0.5)
ax.set_yticklabels(ENMED, fontsize=10)

ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title('Δ vs Qwen3-14B per (model, task, shot)\n'
             'Stripes = Positive Δ (EnMed better), Dots = Negative Δ (EnMed worse)\n'
             'Stars = item-level paired t-test p < 0.05 (*), 0.01 (**), 0.001 (***)',
             fontsize=13)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig08_sig_heatmap.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
rows = []
# Use 'ttest_item' which was defined in the previous cell
for m in ENMED:
    sub = ttest_item[ttest_item['model'] == m]
    rows.append({
        'model':         m,
        'wins/9':        int((sub['delta'] > 0).sum()),
        'sig_wins/9':    int(((sub['delta'] > 0) & (sub[p_col] < 0.05)).sum()),
        'sig_losses/9': int(((sub['delta'] < 0) & (sub[p_col] < 0.05)).sum()),
        'ties/9':        int((sub['delta'].abs() < 0.005).sum()),
        'mean_delta':    sub['delta'].mean(),
        'mean_cohens_d': sub['cohens_d'].mean(),
        'median_p':      sub[p_col].median(),
    })
sig_summary = pd.DataFrame(rows).sort_values('sig_wins/9', ascending=False)
sig_summary.to_csv(os.path.join(OUT_DIR, 'significance_summary.csv'), index=False)
sig_summary.round(4)

In [ ]:
rows = []
for m in ENMED:
    sub = ttest_item[ttest_item['model'] == m]
    rows.append({
        'model':         m,
        'wins/9':        int((sub['delta'] > 0).sum()),
        'sig_wins/9':    int(((sub['delta'] > 0) & (sub[p_col] < 0.05)).sum()),
        'sig_losses/9':  int(((sub['delta'] < 0) & (sub[p_col] < 0.05)).sum()),
        'ties/9':        int((sub['delta'].abs() < 0.005).sum()),
        'mean_delta':    sub['delta'].mean(),
        'mean_cohens_d': sub['cohens_d'].mean(),
        'median_p':      sub[p_col].median(),
    })
sig_summary = pd.DataFrame(rows).sort_values('sig_wins/9', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
order_m = sig_summary.set_index('model')
order_m = order_m.iloc[::-1]   # best on top
y = np.arange(len(order_m))
sig_w = order_m['sig_wins/9'].values
sig_l = order_m['sig_losses/9'].values
nsw   = (order_m['wins/9'] - order_m['sig_wins/9']).values
nsl   = 9 - sig_w - sig_l - nsw

# Pure B&W styling using patterns
ax.barh(y, sig_w,    color='white', hatch='////', edgecolor='black', label='Sig. wins (p<0.05)')
ax.barh(y, nsw,      left=sig_w,            color='white', hatch='\\\\', edgecolor='black', label='Numeric wins')
ax.barh(y, nsl,      left=sig_w+nsw,        color='white', hatch='..', edgecolor='black', label='Numeric losses')
ax.barh(y, sig_l,    left=sig_w+nsw+nsl,    color='white', hatch='xx', edgecolor='black', label='Sig. losses (p<0.05)')
for i, m in enumerate(order_m.index):
    vals = [sig_w[i], nsw[i], nsl[i], sig_l[i]]
    starts = [0, sig_w[i], sig_w[i]+nsw[i], sig_w[i]+nsw[i]+nsl[i]]
    for v, st in zip(vals, starts):
        if v > 0:
            ax.text(st + v/2, i, f"{int(v)}", ha='center', va='center',
                    color='black', fontweight='bold',
                    bbox=dict(facecolor='white', edgecolor='none', pad=1, alpha=0.8))
ax.axvline(4.5, color='black', linestyle=':', linewidth=1.5)
ax.set_yticks(y); ax.set_yticklabels(order_m.index)
ax.set_xlim(0, 9); ax.set_xlabel('# of (task, shot) cells')
ax.set_title('Significance record across 9 independent tests (Pure B&W)\n'
             '(dotted line = majority threshold)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig10_sig_summary.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
best_per_cell = df.idxmax(axis=0)
pivot = pd.DataFrame(index=TASKS, columns=[f"{s}-shot" for s in SHOTS])
for (t, s), m in best_per_cell.items():
    pivot.loc[t, f"{s}-shot"] = m
pivot.to_csv(os.path.join(OUT_DIR, 'best_model_per_cell.csv'))
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
all_winners = sorted(set(pivot.values.flatten()))
winner_to_id = {w: i for i, w in enumerate(all_winners)}

# Distinct hatches for B&W styling
hatches = ['////', '....', '\\\\', 'xx', '++', 'oo'][:len(all_winners)]
winner_to_hatch = {w: hatches[i] for i, w in enumerate(all_winners)}

ax.set_xlim(-0.5, len(SHOTS) - 0.5)
ax.set_ylim(len(TASKS) - 0.5, -0.5)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        winner = pivot.iloc[i, j]
        score  = df.loc[winner, (TASKS[i], SHOTS[j])]

        # Add patterned rectangle
        rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor='white', edgecolor='black',
                             hatch=winner_to_hatch[winner], linewidth=1)
        ax.add_patch(rect)

        # Text with white background for readability
        ax.text(j, i, f"{winner}\n{score:.3f}",
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='black', bbox=dict(facecolor='white', edgecolor='none', pad=2, alpha=0.85))

ax.set_xticks(range(len(SHOTS))); ax.set_xticklabels([f"{s}-shot" for s in SHOTS])
ax.set_yticks(range(len(TASKS))); ax.set_yticklabels([TASK_LABEL[t] for t in TASKS])
ax.set_title('Stage 10 — Best model at every (task, shot) cell (Pure B&W)')

patches = [mpatches.Patch(facecolor='white', edgecolor='black', hatch=winner_to_hatch[w], label=w) for w in all_winners]
ax.legend(handles=patches, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10)

plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig11_best_per_cell.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
# Build a per-model summary from item-level results
if 'p_value' in ttest_item.columns:
    p_col = 'p_value'
else:
    p_col = 'p_bonferroni'

rows = []
for m in ENMED:
    sub = ttest_item[ttest_item['model'] == m].copy()
    sub = sub.dropna(subset=['delta', p_col])
    n_cells = len(sub)
    sig_wins   = int(((sub[p_col] < 0.05) & (sub['delta'] > 0)).sum())
    sig_losses = int(((sub[p_col] < 0.05) & (sub['delta'] < 0)).sum())
    num_wins   = int((sub['delta'] > 0).sum())

    if sig_wins >= 3 and sig_losses == 0:
        verdict_cat = 'sig_better'
        verdict     = f"✓ Significantly better on {sig_wins}/{n_cells} cells, never significantly worse"
    elif sig_wins > sig_losses:
        verdict_cat = 'mixed_better'
        verdict     = f"~ Mixed: wins {sig_wins} cells significantly, loses {sig_losses}"
    elif sig_wins == 0 and sig_losses == 0:
        verdict_cat = 'tied'
        verdict     = f"= No significant cells (numerical wins: {num_wins}/{n_cells})"
    elif sig_losses > sig_wins:
        verdict_cat = 'worse'
        verdict     = f"✗ Significantly worse on {sig_losses} cells, wins only {sig_wins}"
    else:
        verdict_cat = 'mixed'
        verdict     = f"~ Mixed: {sig_wins} sig wins, {sig_losses} sig losses"

    rows.append({
        'model':          m,
        'cells_tested':   n_cells,
        'sig_wins':       sig_wins,
        'sig_losses':     sig_losses,
        'num_wins':       num_wins,
        'mean_delta_norm': sub['delta'].mean(),  # raw scale; tasks not pooled
        'verdict':        verdict,
        '_cat':           verdict_cat,
    })

reviewer_table = pd.DataFrame(rows)
reviewer_table.drop(columns=['_cat']).to_csv(
    os.path.join(OUT_DIR, 'reviewer_self_defense.csv'), index=False)
reviewer_table.drop(columns=['_cat'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
verdict_colors = {'sig_better':'#2ca02c', 'mixed_better':'#a8e6a3',
                  'tied':'#cccccc', 'mixed':'#f5b0b0', 'worse':'#d62728'}

order_rev = reviewer_table.sort_values(
    by=['_cat'],
    key=lambda c: c.map({'sig_better':0,'mixed_better':1,'tied':2,'mixed':3,'worse':4})
).reset_index(drop=True)

y = np.arange(len(order_rev))
sig_wins   = order_rev['sig_wins'].values
sig_losses = order_rev['sig_losses'].values

ax.barh(y, sig_wins,   color='#2ca02c', edgecolor='black', linewidth=0.8, label='Sig. wins (p<0.05)')
ax.barh(y, -sig_losses, color='#d62728', edgecolor='black', linewidth=0.8, label='Sig. losses (p<0.05)')

ax.axvline(0, color='black', linewidth=1.2)
ax.set_yticks(y); ax.set_yticklabels(order_rev['model'])
ax.set_xlabel('# of (task, shot) cells (out of 9)')
ax.set_title('Stage 10 — Reviewer self-defense per EnMed variant\n'
             'Item-level paired t-test, α=0.05 uncorrected')
ax.legend(loc='lower right', fontsize=10)

# Inline annotations
for i, (_, r) in enumerate(order_rev.iterrows()):
    if r['sig_wins'] > 0:
        ax.text(r['sig_wins']/2, i, str(r['sig_wins']),
                color='white', ha='center', va='center', fontweight='bold', fontsize=12)
    if r['sig_losses'] > 0:
        ax.text(-r['sig_losses']/2, i, str(r['sig_losses']),
                color='white', ha='center', va='center', fontweight='bold', fontsize=12)
    # Verdict text to the right
    ax.text(max(sig_wins.max()+0.3, 1), i, r['verdict'],
            va='center', fontsize=9.5, alpha=0.85)

xmax = max(sig_wins.max(), 9)
ax.set_xlim(-(sig_losses.max()+0.5), xmax + 5)
plt.tight_layout()
fp = os.path.join(FIG_DIR, 'fig11_reviewer_verdicts.png')
plt.savefig(fp, bbox_inches='tight'); plt.show()
print(f"✓ {fp}")

In [ ]:
print("═" * 72)
print(f"  NB8 (DEEP STATS vs Qwen3-14B) COMPLETE")
print("═" * 72)
print(f"  CSV outputs : {OUT_DIR}/")
print(f"  Figures     : {FIG_DIR}/")
print()
print("  Headline (item-level paired t-test, 9 independent tests per model):")
for _, r in sig_summary.iterrows():
    print(f"    {r['model']:<16}  "
          f"sig wins: {int(r['sig_wins/9'])}/9  "
          f"sig losses: {int(r['sig_losses/9'])}/9  "
          f"mean Δ: {r['mean_delta']:+.4f}  "
          f"mean d: {r['mean_cohens_d']:+.2f}")
print()
print("  Figures (one per table):")
for f in sorted(os.listdir(FIG_DIR)):
    print(f"    {f}")
print("═" * 72)

## §8 — Export Quantized GGUF Checkpoints *(NEW)*

Exports the trained EnMed-Unified model to GGUF format for use with `llama.cpp`.
Three quantization levels are produced:

| Quant | Size (14B approx) | Recommended for |
|---|---|---|
| `q4_k_m` | ~8.5 GB | **Best CPU choice** — 4-bit, good quality |
| `q8_0` | ~15 GB | Higher fidelity, more RAM needed |
| `bf16` | ~28 GB | Full precision reference |

> **What is new here:** This section is not in any of your 8 notebooks. The `push_to_hub_gguf` call pattern comes from the Qwopus training notebook you provided earlier, adapted for the EnMed-Unified checkpoint.


In [ ]:
# §8 — GGUF export  ★ NEW (not in NB1–NB8)
#
# Re-use the merged EnMed-Unified model already in memory from §4/§5,
# OR re-load it from the Hub before running this cell.
# The pattern (push_to_hub_gguf) is from your Qwopus training notebook.

from unsloth import FastLanguageModel

# ── Re-load if you're running this section in a fresh session ────────────────
# Uncomment and set your HF repo if needed:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name  = cfg.hf_repo_unified,   # e.g. "boods/EnToFrMedicaLLM-Unified"
#     max_seq_length = cfg.max_seq_length,
#     load_in_4bit   = True,
#     token = HF_TOKEN,
# )

GGUF_REPO = f"{cfg.hf_user}/{cfg.project_name}-Unified-GGUF"

# Export + push — produces one HF repo with three quant files.
model.push_to_hub_gguf(
    GGUF_REPO,
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0", "bf16"],
    token = HF_TOKEN,
)

logger.info(f"✓ GGUF pushed → https://huggingface.co/{GGUF_REPO}")
logger.info("  Files:")
logger.info(f"  - {cfg.project_name}-Unified-Q4_K_M.gguf  (~8.5 GB)  ← best for CPU")
logger.info(f"  - {cfg.project_name}-Unified-Q8_0.gguf     (~15 GB)")
logger.info(f"  - {cfg.project_name}-Unified-BF16.gguf     (~28 GB)")


## §9 — CPU Inference with Quantized GGUF *(NEW)*

Runs your trained EnMed-Unified model **on a CPU-only machine** — no GPU, no CUDA, no Unsloth.  
Uses `llama-cpp-python` to load the `q4_k_m` GGUF from §8.

**Requirements:** ~10–12 GB free RAM (for 14B q4_k_m). Works on any laptop.

> **What is new here:** This entire section is new — none of NB1–NB8 covers CPU inference. The prompt templates reuse your exact `SYSTEM_PROMPT_MEDICAL_FR`, `INSTR_MCQA_FR`, `INSTR_EXTQA_FR`, `INSTR_ABSQA_FR` constants from `enmed_core.py`.


In [ ]:
# §9a — Install llama-cpp-python (CPU build, no CUDA)  ★ NEW
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python",        # pure-CPU wheel — no CUDA needed
    "huggingface_hub",
])
print("✓ llama-cpp-python installed")


In [ ]:
# §9b — Download the q4_k_m GGUF from HF Hub  ★ NEW
from huggingface_hub import hf_hub_download

GGUF_REPO = f"{cfg.hf_user}/{cfg.project_name}-Unified-GGUF"
GGUF_FILE = f"{cfg.project_name}-Unified-Q4_K_M.gguf"   # ~8.5 GB

print(f"Downloading {GGUF_FILE} from {GGUF_REPO} …")
gguf_path = hf_hub_download(
    repo_id  = GGUF_REPO,
    filename = GGUF_FILE,
    token    = HF_TOKEN,
)
print(f"✓ Downloaded → {gguf_path}")


In [ ]:
# §9c — Load on CPU with llama-cpp-python  ★ NEW
from llama_cpp import Llama

llm = Llama(
    model_path   = gguf_path,
    n_ctx        = 2048,    # context window
    n_threads    = None,    # None = all CPU cores
    n_gpu_layers = 0,       # 0 = pure CPU
    verbose      = False,
)
print(f"✓ {GGUF_FILE} loaded on CPU")
print(f"  n_ctx={llm.context_params.n_ctx}  threads={llm.context_params.n_threads}")


In [ ]:
# §9d — MCQA inference on CPU (same prompt format as NB5a runner)  ★ NEW
#
# Reuses your ec.SYSTEM_PROMPT_MEDICAL_FR and ec.INSTR_MCQA_FR constants
# so the prompt is identical to what was used during evaluation.

question = "Quelle est la principale cause d'insuffisance rénale aiguë en réanimation ?"
options  = {"A": "Glomérulonéphrite aiguë",
            "B": "Nécrose tubulaire aiguë ischémique",
            "C": "Pyélonéphrite aiguë",
            "D": "Lithiase urinaire"}
opts_str = "\n".join(f"{k}) {v}" for k, v in options.items())

messages = [
    {"role": "system",  "content": ec.SYSTEM_PROMPT_MEDICAL_FR},
    {"role": "user",    "content": (
        f"{ec.INSTR_MCQA_FR}\n\n"
        f"Question: {question}\n"
        f"{opts_str}\n"
        "Réponse:"
    )},
]

out = llm.create_chat_completion(
    messages    = messages,
    max_tokens  = 8,
    temperature = 0.0,   # deterministic — matches eval protocol
)
pred = out["choices"][0]["message"]["content"].strip()
print(f"Prediction : {pred}")
print(f"Correct    : B (Nécrose tubulaire aiguë ischémique)")


In [ ]:
# §9e — ExtQA inference on CPU  ★ NEW
context  = ("Le patient présente une oligurie progressive depuis 48 heures suite "
            "à un choc septique réfractaire traité en réanimation. La créatinine "
            "est à 450 µmol/L. L'échographie rénale montre des reins de taille normale "
            "sans obstacle.")
question = "Quelle est la durée de l'oligurie ?"

messages = [
    {"role": "system",  "content": ec.SYSTEM_PROMPT_MEDICAL_FR},
    {"role": "user",    "content": (
        f"{ec.INSTR_EXTQA_FR}\n\n"
        f"Texte:\n{context}\n\n"
        f"Question: {question}\n"
        "Réponse:"
    )},
]

out = llm.create_chat_completion(messages=messages, max_tokens=32, temperature=0.0)
pred = out["choices"][0]["message"]["content"].strip()
print(f"Prediction : {pred}")
print(f"Gold span  : 48 heures")


In [ ]:
# §9f — AbsQA inference on CPU  ★ NEW
question = "Quels sont les signes cliniques et biologiques d'une insuffisance rénale aiguë ?"

messages = [
    {"role": "system",  "content": ec.SYSTEM_PROMPT_MEDICAL_FR},
    {"role": "user",    "content": (
        f"{ec.INSTR_ABSQA_FR}\n\n"
        f"Question: {question}\n"
        "Réponse:"
    )},
]

out = llm.create_chat_completion(messages=messages, max_tokens=300, temperature=0.7)
print("AbsQA response:")
print(out["choices"][0]["message"]["content"].strip())


In [ ]:
# §9g — Batch evaluation on the NB1 test set (CPU, no GPU)  ★ NEW
#
# Runs MCQA accuracy on your locked test set entirely on CPU.
# Useful for reproducing results on hardware without a GPU.
import json, re
from tqdm.auto import tqdm

TEST_PATH = os.path.join(cfg.results_dir, "test.jsonl")
if not os.path.exists(TEST_PATH):
    print(f"Run §2 first to generate {TEST_PATH}")
else:
    test_rows = ec.load_jsonl(TEST_PATH)
    mcqa_test = [r for r in test_rows if r["task"] == "mcqa"][:50]  # cap at 50 for demo

    correct = 0
    for ex in tqdm(mcqa_test, desc="CPU MCQA eval"):
        opts = ex.get("options") or {}
        if isinstance(opts, list):
            opts = {chr(65+i): str(o) for i,o in enumerate(opts)}
        opts_str = "\n".join(f"{k}) {v}" for k,v in opts.items())
        msgs = [
            {"role":"system",  "content": ec.SYSTEM_PROMPT_MEDICAL_FR},
            {"role":"user",    "content": (
                f"{ec.INSTR_MCQA_FR}\n\nQuestion: {ex['question']}\n{opts_str}\nRéponse:"
            )},
        ]
        out  = llm.create_chat_completion(msgs, max_tokens=8, temperature=0.0)
        pred = ec.parse_mcqa_letters(out["choices"][0]["message"]["content"], n_choices=len(opts))
        gold = sorted(set(c for c in (ex["answer"] or "").split(",") if c.strip().isalpha()))
        correct += (set(pred) == set(gold))

    acc = correct / len(mcqa_test) if mcqa_test else 0
    print(f"\nCPU MCQA accuracy ({len(mcqa_test)} items): {acc:.4f}")


### §9 notes

- Replace `n_gpu_layers=0` with `n_gpu_layers=-1` if you have a GPU — `llama.cpp` will offload all layers to it and run much faster.
- For a chat server / CLI instead of Python, use [Ollama](https://ollama.com): `ollama create enmed -f Modelfile` where the Modelfile points at the GGUF.
- The `q4_k_m` 14B model needs ~10–12 GB RAM. Use `Qwen3-8B` if RAM is tight.
